# Half Polyomino Ranked Search

Run the cells from top to bottom. The ranked search can write a near-miss gallery, and the direct search verifies candidates without CP-SAT.

In [ ]:
%%writefile requirements.txt
ortools>=9.10


In [ ]:
%%writefile generate_half_polyomino.py
#!/usr/bin/env python3
"""Generate half-cell polyomino-style tiling puzzles.

The puzzle uses a small-cell grid where one ordinary cell is a 2x2 block.
Every ordinary cell in every board/piece shape must be one of six legal masks:

    bit order: 0bTL TR BL BR
      TL = top-left, TR = top-right, BL = bottom-left, BR = bottom-right

That makes these masks legal:

    0000 empty
    1111 full
    1100 top half
    0011 bottom half
    1010 left half
    0101 right half

All 1/4-cell, 3/4-cell, diagonal-half, and L-shaped artifacts are rejected.
"""

from __future__ import annotations

import argparse
import json
import math
import random
import sys
import time
from collections import defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable


SCALE = 2
PIECE_COUNT = 6
PIECE_AREA_SMALL = 16

BOARD_W = 10  # ordinary cells
BOARD_H = 7   # ordinary cells

ALLOW_ROTATE = True
ALLOW_MIRROR = False

TARGET_MIN_SOLUTIONS = 4
TARGET_MAX_SOLUTIONS = 16
SOLUTION_COUNT_LIMIT = 100

# Bit order is 0bTLTRBLBR:
# 0b1000 top-left, 0b0100 top-right, 0b0010 bottom-left, 0b0001 bottom-right.
MASK_EMPTY = 0b0000
MASK_FULL = 0b1111
MASK_TOP = 0b1100
MASK_BOTTOM = 0b0011
MASK_LEFT = 0b1010
MASK_RIGHT = 0b0101

ALLOWED_MASKS = {
    MASK_EMPTY,   # empty
    MASK_FULL,    # full
    MASK_TOP,     # top half
    MASK_BOTTOM,  # bottom half
    MASK_LEFT,    # left half
    MASK_RIGHT,   # right half
}
HALF_MASKS = {MASK_TOP, MASK_BOTTOM, MASK_LEFT, MASK_RIGHT}

MASK_TO_OFFSETS = {
    MASK_EMPTY: (),
    MASK_FULL: ((0, 0), (1, 0), (0, 1), (1, 1)),
    MASK_TOP: ((0, 0), (1, 0)),
    MASK_BOTTOM: ((0, 1), (1, 1)),
    MASK_LEFT: ((0, 0), (0, 1)),
    MASK_RIGHT: ((1, 0), (1, 1)),
}

LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
COLORS = [
    "#df5b57",
    "#4f8bd6",
    "#55a868",
    "#c77cce",
    "#d79542",
    "#4db6ac",
    "#8d6e63",
    "#9e9ac8",
    "#e377c2",
    "#7f7f7f",
    "#bcbd22",
    "#17becf",
]

GUIDED_SUPER_LAYOUTS: list[list[MacroCell]] = [
    # Eight 2x2 ordinary-cell blocks arranged as a non-rectangular board.
    # This layout is deliberately lumpy but still easy to partition cleanly.
    [(1, 0), (2, 0), (3, 0), (0, 1), (1, 1), (2, 1), (1, 2), (2, 2)],
    [(0, 0), (1, 0), (2, 0), (0, 1), (1, 1), (2, 1), (3, 1), (1, 2)],
    [(1, 0), (2, 0), (0, 1), (1, 1), (2, 1), (3, 1), (1, 2), (2, 2)],
    [(0, 0), (1, 0), (2, 0), (1, 1), (2, 1), (3, 1), (2, 2), (3, 2)],
]

ROBUST_PAIR_SHAPE_TEXTS = [
    # All of these are 16 small cells, legal half-cell masks only, connected,
    # and pass the paper-strength checks.  They are intentionally chunky:
    # no one-small-cell bridge and no dangling small-cell tips.
    "##../####/####/####/##..",
    "....##/..####/######/####..",
    ".##./.##./####/####/..##/..##",
    "###./###./####/####/..##",
    "##.../##.../#####/#####/..##.",
    "####/####/####/..##/..##",
    "..##/..##/..##/####/####/..##",
    "..##/####/####/####/..##",
]


Cell = tuple[int, int]
MacroCell = tuple[int, int]


@dataclass(frozen=True)
class Placement:
    piece_index: int
    cells: frozenset[Cell]
    mask: int
    origin: Cell
    orientation: int


@dataclass
class Analysis:
    solution_count: int
    rotated_solution_count: int
    fixed_pieces: int
    movable_pieces: int
    average_piece_candidates: float
    half_cell_count_per_piece: list[int]
    total_half_cell_count: int
    quarter_artifact_count: int
    fragile_artifact_count: int
    duplicate_piece_count: int
    difficulty_score: float


@dataclass
class Candidate:
    board: set[Cell]
    pieces: list[set[Cell]]
    solutions: list[dict[int, frozenset[Cell]]]
    solution_count: int
    placements_by_piece: list[list[Placement]]
    score: float
    analysis: Analysis
    attempts: int


def mask_for_offsets(offsets: Iterable[Cell]) -> int:
    mask = 0
    for dx, dy in offsets:
        if dx == 0 and dy == 0:
            mask |= 0b1000
        elif dx == 1 and dy == 0:
            mask |= 0b0100
        elif dx == 0 and dy == 1:
            mask |= 0b0010
        elif dx == 1 and dy == 1:
            mask |= 0b0001
        else:
            raise ValueError(f"offset outside 2x2 cell: {(dx, dy)}")
    return mask


def cells_to_masks(cells: set[Cell] | frozenset[Cell]) -> dict[MacroCell, int]:
    masks: dict[MacroCell, int] = defaultdict(int)
    for x, y in cells:
        mx, dx = divmod(x, SCALE)
        my, dy = divmod(y, SCALE)
        masks[(mx, my)] |= mask_for_offsets(((dx, dy),))
    return dict(masks)


def masks_to_cells(masks: dict[MacroCell, int]) -> set[Cell]:
    cells: set[Cell] = set()
    for (mx, my), mask in masks.items():
        for dx, dy in MASK_TO_OFFSETS[mask]:
            cells.add((mx * SCALE + dx, my * SCALE + dy))
    return cells


def cells_from_ascii(text: str) -> set[Cell]:
    rows = text.split("/")
    return {
        (x, y)
        for y, row in enumerate(rows)
        for x, char in enumerate(row)
        if char == "#"
    }


def robust_pair_shape_library() -> list[set[Cell]]:
    shapes = [normalize_cells(cells_from_ascii(text)) for text in ROBUST_PAIR_SHAPE_TEXTS]
    valid_shapes = []
    for shape in shapes:
        if (
            len(shape) == PIECE_AREA_SMALL
            and is_legal_half_cell_shape(shape)
            and is_connected(shape)
            and count_half_cells(shape) >= 2
            and count_fragile_artifacts(shape) == 0
        ):
            valid_shapes.append(shape)
    return valid_shapes


def is_legal_half_cell_shape(cells: set[Cell] | frozenset[Cell]) -> bool:
    """
    Check that a small-cell shape contains no 1/4-cell or 3/4-cell artifacts.

    Each ordinary 2x2 cell is converted to a mask.  Only the six masks in
    ALLOWED_MASKS are accepted: empty, full, top, bottom, left, and right.
    """
    return all(mask in ALLOWED_MASKS for mask in cells_to_masks(cells).values())


def count_half_cells(cells: set[Cell] | frozenset[Cell]) -> int:
    """
    Return how many ordinary cells are occupied by a legal half-cell mask.
    """
    return sum(1 for mask in cells_to_masks(cells).values() if mask in HALF_MASKS)


def count_quarter_artifacts(cells: set[Cell] | frozenset[Cell]) -> int:
    """Count ordinary cells whose 2x2 mask is not one of the legal masks."""
    return sum(1 for mask in cells_to_masks(cells).values() if mask not in ALLOWED_MASKS)


def has_quarter_artifact(cells: set[Cell] | frozenset[Cell]) -> bool:
    """
    Return True if a shape contains 1/4-cell, 3/4-cell, diagonal-half, or L masks.
    """
    return count_quarter_artifacts(cells) > 0


def small_cell_degree(cells: set[Cell] | frozenset[Cell], cell: Cell) -> int:
    return sum(1 for neighbor in neighbors4(cell) if neighbor in cells)


def has_dangling_small_cell(cells: set[Cell] | frozenset[Cell]) -> bool:
    """Reject paper-weak tips that hang from a single small-cell edge."""
    return any(small_cell_degree(cells, cell) <= 1 for cell in cells)


def has_articulation_small_cell(cells: set[Cell] | frozenset[Cell]) -> bool:
    """Reject shapes where one small-cell is the only bridge between regions."""
    if len(cells) <= 2:
        return False
    original = set(cells)
    for cell in original:
        remaining = original - {cell}
        if remaining and not is_connected(remaining):
            return True
    return False


def has_repeated_half_strip(cells: set[Cell] | frozenset[Cell]) -> bool:
    """
    Reject consecutive half-cell masks that make a half-cell-wide strip.

    A single half-cell tab/notch is acceptable.  Two or more left/right half masks
    stacked vertically, or two or more top/bottom half masks chained horizontally,
    create a paper-thin band and are rejected.
    """
    masks = cells_to_masks(cells)
    for (mx, my), mask in masks.items():
        if mask in (MASK_LEFT, MASK_RIGHT):
            if masks.get((mx, my - 1)) == mask or masks.get((mx, my + 1)) == mask:
                return True
        if mask in (MASK_TOP, MASK_BOTTOM):
            if masks.get((mx - 1, my)) == mask or masks.get((mx + 1, my)) == mask:
                return True
    return False


def count_fragile_artifacts(cells: set[Cell] | frozenset[Cell]) -> int:
    """
    Count paper-weak shape features.

    This is deliberately conservative: paper pieces should not have narrow
    half-cell strings, dangling small-cell ends, or a one-small-cell bridge.
    """
    count = 0
    if has_dangling_small_cell(cells):
        count += 1
    if has_articulation_small_cell(cells):
        count += 1
    if has_repeated_half_strip(cells):
        count += 1
    return count


def duplicate_piece_count(pieces: list[set[Cell]]) -> int:
    signatures = [canonical_signature(piece) for piece in pieces]
    return len(signatures) - len(set(signatures))


def solution_identity_signature(
    solution: dict[int, frozenset[Cell]],
    pieces: list[set[Cell]],
) -> tuple[tuple[tuple[Cell, ...], tuple[tuple[Cell, ...], ...]], ...]:
    """Canonicalize a solution modulo swaps of identical physical cuts."""
    groups: dict[tuple[Cell, ...], list[tuple[Cell, ...]]] = defaultdict(list)
    for piece_index, cells in solution.items():
        piece_sig = canonical_signature(pieces[piece_index])
        groups[piece_sig].append(tuple(sorted(cells)))
    return tuple(
        (piece_sig, tuple(sorted(placements)))
        for piece_sig, placements in sorted(groups.items())
    )


def count_effective_solutions(
    solutions: list[dict[int, frozenset[Cell]]],
    pieces: list[set[Cell]],
) -> int:
    """Count solutions after ignoring pure swaps of identical pieces."""
    return len({solution_identity_signature(solution, pieces) for solution in solutions})


def normalize_cells(cells: set[Cell] | frozenset[Cell]) -> set[Cell]:
    if not cells:
        return set()
    min_x = min(x for x, _ in cells)
    min_y = min(y for _, y in cells)
    return {(x - min_x, y - min_y) for x, y in cells}


def bounds(cells: set[Cell] | frozenset[Cell]) -> tuple[int, int, int, int]:
    min_x = min(x for x, _ in cells)
    max_x = max(x for x, _ in cells)
    min_y = min(y for _, y in cells)
    max_y = max(y for _, y in cells)
    return min_x, min_y, max_x, max_y


def is_connected(cells: set[Cell] | frozenset[Cell]) -> bool:
    if not cells:
        return False
    start = next(iter(cells))
    seen = {start}
    queue = deque([start])
    while queue:
        x, y = queue.popleft()
        for nx, ny in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
            if (nx, ny) in cells and (nx, ny) not in seen:
                seen.add((nx, ny))
                queue.append((nx, ny))
    return len(seen) == len(cells)


def is_macro_connected(cells: set[MacroCell] | frozenset[MacroCell]) -> bool:
    if not cells:
        return False
    start = next(iter(cells))
    seen = {start}
    queue = deque([start])
    while queue:
        x, y = queue.popleft()
        for nx, ny in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
            if (nx, ny) in cells and (nx, ny) not in seen:
                seen.add((nx, ny))
                queue.append((nx, ny))
    return len(seen) == len(cells)


def has_macro_hole(cells: set[MacroCell]) -> bool:
    min_x = min(x for x, _ in cells) - 1
    max_x = max(x for x, _ in cells) + 1
    min_y = min(y for _, y in cells) - 1
    max_y = max(y for _, y in cells) + 1
    start = (min_x, min_y)
    outside = {start}
    queue = deque([start])
    while queue:
        x, y = queue.popleft()
        for nx, ny in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
            if not (min_x <= nx <= max_x and min_y <= ny <= max_y):
                continue
            if (nx, ny) in cells or (nx, ny) in outside:
                continue
            outside.add((nx, ny))
            queue.append((nx, ny))
    for y in range(min_y + 1, max_y):
        for x in range(min_x + 1, max_x):
            if (x, y) not in cells and (x, y) not in outside:
                return True
    return False


def has_small_hole(cells: set[Cell]) -> bool:
    min_x = min(x for x, _ in cells) - 1
    max_x = max(x for x, _ in cells) + 1
    min_y = min(y for _, y in cells) - 1
    max_y = max(y for _, y in cells) + 1
    start = (min_x, min_y)
    outside = {start}
    queue = deque([start])
    while queue:
        x, y = queue.popleft()
        for nx, ny in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
            if not (min_x <= nx <= max_x and min_y <= ny <= max_y):
                continue
            if (nx, ny) in cells or (nx, ny) in outside:
                continue
            outside.add((nx, ny))
            queue.append((nx, ny))
    for y in range(min_y + 1, max_y):
        for x in range(min_x + 1, max_x):
            if (x, y) not in cells and (x, y) not in outside:
                return True
    return False


def macro_to_full_small(cells: set[MacroCell] | frozenset[MacroCell]) -> set[Cell]:
    masks = {cell: MASK_FULL for cell in cells}
    return masks_to_cells(masks)


def random_macro_board(
    rng: random.Random,
    width: int,
    height: int,
    area: int,
    allow_holes: bool,
    max_tries: int = 500,
) -> set[MacroCell] | None:
    if area > width * height:
        return None

    center = (width // 2, height // 2)
    all_cells = {(x, y) for y in range(height) for x in range(width)}
    for _ in range(max_tries):
        start = (
            min(width - 1, max(0, center[0] + rng.randint(-2, 2))),
            min(height - 1, max(0, center[1] + rng.randint(-1, 1))),
        )
        cells = {start}
        frontier = {
            (nx, ny)
            for nx, ny in neighbors4(start)
            if 0 <= nx < width and 0 <= ny < height
        }

        while len(cells) < area and frontier:
            # Prefer compact-but-lumpy growth near existing cells.
            candidates = list(frontier)
            weights = []
            for c in candidates:
                neighbor_count = sum(1 for n in neighbors4(c) if n in cells)
                distance = abs(c[0] - center[0]) + abs(c[1] - center[1])
                weights.append(1.0 + neighbor_count * 3.0 + max(0, 8 - distance) * 0.12)
            chosen = rng.choices(candidates, weights=weights, k=1)[0]
            frontier.remove(chosen)
            cells.add(chosen)
            for n in neighbors4(chosen):
                if n not in cells and n in all_cells:
                    frontier.add(n)

        if len(cells) != area:
            continue
        if not is_macro_connected(cells):
            continue
        if not allow_holes and has_macro_hole(cells):
            continue
        min_x = min(x for x, _ in cells)
        max_x = max(x for x, _ in cells)
        min_y = min(y for _, y in cells)
        max_y = max(y for _, y in cells)
        bbox_w = max_x - min_x + 1
        bbox_h = max_y - min_y + 1
        if bbox_w < 5 or bbox_h < 4:
            continue
        if bbox_w * bbox_h == area:
            continue
        fill_ratio = area / (bbox_w * bbox_h)
        if fill_ratio < 0.48 or fill_ratio > 0.88:
            continue
        return cells
    return None


def neighbors4(cell: Cell) -> tuple[Cell, Cell, Cell, Cell]:
    x, y = cell
    return ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1))


def connected_subsets_of_size(
    board: set[MacroCell],
    size: int = 4,
) -> list[frozenset[MacroCell]]:
    """Enumerate connected macro-cell subsets of a given size."""
    subsets: set[frozenset[MacroCell]] = set()
    ordered = sorted(board)
    for start in ordered:
        stack: list[tuple[frozenset[MacroCell], frozenset[MacroCell]]] = [
            (frozenset({start}), frozenset(n for n in neighbors4(start) if n in board))
        ]
        while stack:
            shape, frontier = stack.pop()
            if len(shape) == size:
                subsets.add(shape)
                continue
            if not frontier:
                continue
            for cell in list(frontier):
                new_shape = set(shape)
                new_shape.add(cell)
                # This ordering guard avoids producing every subset from every
                # possible start while still allowing non-monotone shapes.
                if min(new_shape) != start:
                    continue
                new_frontier = set(frontier)
                new_frontier.remove(cell)
                for n in neighbors4(cell):
                    if n in board and n not in new_shape:
                        new_frontier.add(n)
                stack.append((frozenset(new_shape), frozenset(new_frontier)))
    return sorted(subsets, key=lambda s: (min(s), sorted(s)))


def partition_into_tetrominoes(
    board: set[MacroCell],
    piece_count: int,
    rng: random.Random,
    max_nodes: int = 30_000,
) -> list[set[MacroCell]] | None:
    subsets = connected_subsets_of_size(board, 4)
    by_cell: dict[MacroCell, list[frozenset[MacroCell]]] = defaultdict(list)
    for subset in subsets:
        for cell in subset:
            by_cell[cell].append(subset)
    for items in by_cell.values():
        rng.shuffle(items)

    nodes = 0

    def search(remaining: frozenset[MacroCell], chosen: list[frozenset[MacroCell]]):
        nonlocal nodes
        nodes += 1
        if nodes > max_nodes:
            return None
        if not remaining:
            return chosen
        if len(chosen) >= piece_count:
            return None

        best_cell = None
        best_options: list[frozenset[MacroCell]] | None = None
        for cell in remaining:
            options = [s for s in by_cell[cell] if s <= remaining]
            if not options:
                return None
            if best_options is None or len(options) < len(best_options):
                best_cell = cell
                best_options = options
                if len(options) == 1:
                    break
        assert best_cell is not None and best_options is not None
        options = best_options[:]
        rng.shuffle(options)
        for subset in options:
            result = search(frozenset(remaining - subset), chosen + [subset])
            if result is not None:
                return result
        return None

    result = search(frozenset(board), [])
    if result is None or len(result) != piece_count:
        return None
    return [set(s) for s in result]


def region_map(regions: list[set[MacroCell]]) -> dict[MacroCell, int]:
    mapping: dict[MacroCell, int] = {}
    for index, region in enumerate(regions):
        for cell in region:
            mapping[cell] = index
    return mapping


def make_swapped_pieces(
    regions: list[set[MacroCell]],
    rng: random.Random,
    min_half_cells: int = 2,
    max_tries: int = 250,
) -> list[set[Cell]] | None:
    mapping = region_map(regions)
    edges: list[tuple[MacroCell, MacroCell, int, int, str]] = []
    for cell, p in mapping.items():
        x, y = cell
        for n, direction in (((x + 1, y), "h"), ((x, y + 1), "v")):
            q = mapping.get(n)
            if q is not None and q != p:
                edges.append((cell, n, p, q, direction))

    if not edges:
        return None

    for _ in range(max_tries):
        rng.shuffle(edges)
        selected: list[tuple[MacroCell, MacroCell, int, int, str, bool]] = []
        used_cells: set[MacroCell] = set()
        covered: set[int] = set()

        # First cover every piece with at least one half-cell swap.
        for c, d, p, q, direction in edges:
            if c in used_cells or d in used_cells:
                continue
            if p in covered and q in covered:
                continue
            selected.append((c, d, p, q, direction, rng.choice((False, True))))
            used_cells.add(c)
            used_cells.add(d)
            covered.add(p)
            covered.add(q)
            if len(covered) == len(regions):
                break

        if len(covered) != len(regions):
            continue

        # Add a few extra legal-looking offsets, still avoiding cell conflicts.
        for c, d, p, q, direction in edges:
            if c in used_cells or d in used_cells:
                continue
            if rng.random() > 0.22:
                continue
            selected.append((c, d, p, q, direction, rng.choice((False, True))))
            used_cells.add(c)
            used_cells.add(d)

        masks_by_piece: list[dict[MacroCell, int]] = []
        for region in regions:
            masks_by_piece.append({cell: MASK_FULL for cell in region})

        for c, d, p, q, direction, flip in selected:
            if direction == "h":
                p_mask, q_mask = (MASK_TOP, MASK_BOTTOM) if not flip else (MASK_BOTTOM, MASK_TOP)
            else:
                p_mask, q_mask = (MASK_LEFT, MASK_RIGHT) if not flip else (MASK_RIGHT, MASK_LEFT)
            masks_by_piece[p][c] = p_mask
            masks_by_piece[p][d] = p_mask
            masks_by_piece[q][c] = q_mask
            masks_by_piece[q][d] = q_mask

        pieces = [masks_to_cells(masks) for masks in masks_by_piece]
        if all(validate_piece(piece, min_half_cells=min_half_cells) for piece in pieces):
            canonical = [canonical_signature(piece) for piece in pieces]
            # Exact duplicates make solution counting less meaningful for a puzzle.
            if len(set(canonical)) < max(2, len(canonical) - 1):
                continue
            return [normalize_cells(piece) for piece in pieces]
    return None


def validate_piece(piece: set[Cell], min_half_cells: int = 2) -> bool:
    if len(piece) != PIECE_AREA_SMALL:
        return False
    if not is_connected(piece):
        return False
    if not is_legal_half_cell_shape(piece):
        return False
    if count_half_cells(piece) < min_half_cells:
        return False
    if count_fragile_artifacts(piece) != 0:
        return False
    min_x, min_y, max_x, max_y = bounds(piece)
    width = max_x - min_x + 1
    height = max_y - min_y + 1
    if max(width, height) > 12 and min(width, height) <= 2:
        return False
    return True


def transform_cells(
    cells: set[Cell] | frozenset[Cell],
    rotation: int,
    mirror: bool = False,
) -> set[Cell]:
    transformed = set()
    for x, y in cells:
        tx = -x if mirror else x
        ty = y
        r = rotation % 4
        if r == 0:
            nx, ny = tx, ty
        elif r == 1:
            nx, ny = ty, -tx
        elif r == 2:
            nx, ny = -tx, -ty
        else:
            nx, ny = -ty, tx
        transformed.add((nx, ny))
    return normalize_cells(transformed)


def orientations(
    cells: set[Cell] | frozenset[Cell],
    allow_rotate: bool,
    allow_mirror: bool,
) -> list[set[Cell]]:
    rotations = range(4) if allow_rotate else range(1)
    mirrors = (False, True) if allow_mirror else (False,)
    seen: set[tuple[Cell, ...]] = set()
    result: list[set[Cell]] = []
    for mirror in mirrors:
        for rotation in rotations:
            oriented = transform_cells(cells, rotation, mirror)
            if not is_legal_half_cell_shape(oriented):
                continue
            sig = tuple(sorted(oriented))
            if sig in seen:
                continue
            seen.add(sig)
            result.append(oriented)
    return result


def canonical_signature(cells: set[Cell] | frozenset[Cell]) -> tuple[Cell, ...]:
    variants = orientations(cells, allow_rotate=True, allow_mirror=False)
    if not variants:
        return tuple(sorted(normalize_cells(cells)))
    return min(tuple(sorted(v)) for v in variants)


def normalize_board_and_pieces(
    board: set[Cell],
    pieces: list[set[Cell]],
) -> tuple[set[Cell], list[set[Cell]]]:
    min_x, min_y, _, _ = bounds(board)
    board2 = {(x - min_x, y - min_y) for x, y in board}
    return board2, [normalize_cells(piece) for piece in pieces]


def prefer_landscape(
    board: set[Cell],
    pieces: list[set[Cell]],
) -> tuple[set[Cell], list[set[Cell]]]:
    min_x, min_y, max_x, max_y = bounds(board)
    if (max_y - min_y) <= (max_x - min_x):
        return board, pieces
    rotated_board = transform_cells(board, 1, False)
    rotated_pieces = [transform_cells(piece, 1, False) for piece in pieces]
    return normalize_board_and_pieces(rotated_board, rotated_pieces)


def enumerate_placements(
    board: set[Cell],
    pieces: list[set[Cell]],
    allow_rotate: bool,
    allow_mirror: bool,
) -> tuple[list[list[Placement]], dict[Cell, int], int]:
    board_cells = sorted(board)
    index = {cell: i for i, cell in enumerate(board_cells)}
    board_mask = (1 << len(board_cells)) - 1
    _, _, board_max_x, board_max_y = bounds(board)

    placements_by_piece: list[list[Placement]] = []
    for piece_index, piece in enumerate(pieces):
        piece_placements: list[Placement] = []
        for orientation_index, oriented in enumerate(orientations(piece, allow_rotate, allow_mirror)):
            _, _, piece_max_x, piece_max_y = bounds(oriented)
            for ty in range(0, board_max_y - piece_max_y + 1, SCALE):
                for tx in range(0, board_max_x - piece_max_x + 1, SCALE):
                    placed = frozenset((x + tx, y + ty) for x, y in oriented)
                    if not placed <= board:
                        continue
                    mask = 0
                    for cell in placed:
                        mask |= 1 << index[cell]
                    piece_placements.append(
                        Placement(
                            piece_index=piece_index,
                            cells=placed,
                            mask=mask,
                            origin=(tx, ty),
                            orientation=orientation_index,
                        )
                    )
        # Stable de-duplication can matter for symmetric pieces.
        unique: dict[int, Placement] = {}
        for placement in piece_placements:
            unique.setdefault(placement.mask, placement)
        placements_by_piece.append(list(unique.values()))
    return placements_by_piece, index, board_mask


def count_solutions(
    board: set[Cell],
    pieces: list[set[Cell]],
    allow_rotate: bool,
    allow_mirror: bool,
    limit: int,
) -> tuple[int, list[dict[int, frozenset[Cell]]], list[list[Placement]]]:
    placements_by_piece, index, board_mask = enumerate_placements(
        board, pieces, allow_rotate, allow_mirror
    )
    if any(not placements for placements in placements_by_piece):
        return 0, [], placements_by_piece

    cell_to_placements: dict[int, list[tuple[int, Placement]]] = defaultdict(list)
    for piece_index, placements in enumerate(placements_by_piece):
        for placement in placements:
            m = placement.mask
            while m:
                bit = m & -m
                cell_index = bit.bit_length() - 1
                cell_to_placements[cell_index].append((piece_index, placement))
                m ^= bit

    remaining_start = frozenset(range(len(pieces)))
    solutions: list[dict[int, frozenset[Cell]]] = []
    count = 0

    def search(occupied: int, remaining: frozenset[int], chosen: dict[int, Placement]) -> None:
        nonlocal count
        if count >= limit:
            return
        if not remaining:
            if occupied == board_mask:
                count += 1
                if len(solutions) < limit:
                    solutions.append({p: placement.cells for p, placement in chosen.items()})
            return

        empty_mask = board_mask & ~occupied
        best_cell_index = None
        best_options: list[tuple[int, Placement]] | None = None
        m = empty_mask
        while m:
            bit = m & -m
            cell_index = bit.bit_length() - 1
            options = [
                (p, placement)
                for p, placement in cell_to_placements[cell_index]
                if p in remaining and (placement.mask & occupied) == 0
            ]
            if not options:
                return
            if best_options is None or len(options) < len(best_options):
                best_cell_index = cell_index
                best_options = options
                if len(options) == 1:
                    break
            m ^= bit

        assert best_cell_index is not None and best_options is not None
        # Deterministic order keeps seeded runs reproducible.
        best_options.sort(key=lambda item: (item[0], item[1].origin, item[1].orientation))
        for piece_index, placement in best_options:
            chosen[piece_index] = placement
            search(
                occupied | placement.mask,
                frozenset(p for p in remaining if p != piece_index),
                chosen,
            )
            chosen.pop(piece_index, None)
            if count >= limit:
                return

    search(0, remaining_start, {})
    return count, solutions, placements_by_piece


def analyze_candidate(
    board: set[Cell],
    pieces: list[set[Cell]],
    solutions: list[dict[int, frozenset[Cell]]],
    solution_count: int,
    rotated_solution_count: int,
    placements_by_piece: list[list[Placement]],
    min_solutions: int,
    max_solutions: int,
) -> Analysis:
    half_counts = [count_half_cells(piece) for piece in pieces]
    total_half = sum(half_counts)
    quarter_count = count_quarter_artifacts(board) + sum(
        count_quarter_artifacts(piece) for piece in pieces
    )
    fragile_count = sum(count_fragile_artifacts(piece) for piece in pieces)
    duplicate_count = duplicate_piece_count(pieces)
    fixed = 0
    if solutions:
        for piece_index in range(len(pieces)):
            seen_positions = {solution[piece_index] for solution in solutions if piece_index in solution}
            if len(seen_positions) == 1:
                fixed += 1
    movable = len(pieces) - fixed
    avg_candidates = sum(len(p) for p in placements_by_piece) / max(1, len(placements_by_piece))
    score = score_candidate(
        board=board,
        pieces=pieces,
        solution_count=solution_count,
        min_solutions=min_solutions,
        max_solutions=max_solutions,
        fixed_pieces=fixed,
        average_piece_candidates=avg_candidates,
        total_half_cell_count=total_half,
        quarter_artifact_count=quarter_count,
        fragile_artifact_count=fragile_count,
        duplicate_piece_count=duplicate_count,
    )
    return Analysis(
        solution_count=solution_count,
        rotated_solution_count=rotated_solution_count,
        fixed_pieces=fixed,
        movable_pieces=movable,
        average_piece_candidates=avg_candidates,
        half_cell_count_per_piece=half_counts,
        total_half_cell_count=total_half,
        quarter_artifact_count=quarter_count,
        fragile_artifact_count=fragile_count,
        duplicate_piece_count=duplicate_count,
        difficulty_score=score,
    )


def score_candidate(
    board: set[Cell],
    pieces: list[set[Cell]],
    solution_count: int,
    min_solutions: int,
    max_solutions: int,
    fixed_pieces: int,
    average_piece_candidates: float,
    total_half_cell_count: int,
    quarter_artifact_count: int,
    fragile_artifact_count: int,
    duplicate_piece_count: int,
) -> float:
    if quarter_artifact_count:
        return -10_000.0 - quarter_artifact_count * 1000
    if fragile_artifact_count:
        return -8_000.0 - fragile_artifact_count * 1000
    if duplicate_piece_count:
        return -6_000.0 - duplicate_piece_count * 1000
    target_mid = (min_solutions + max_solutions) / 2
    score = 120.0 - abs(solution_count - target_mid) * 5.0
    score += min(total_half_cell_count, len(pieces) * 4) * 4.0
    score += max(0, len(pieces) - fixed_pieces) * 8.0

    macro_board = {(x // SCALE, y // SCALE) for x, y in board}
    min_x = min(x for x, _ in macro_board)
    max_x = max(x for x, _ in macro_board)
    min_y = min(y for _, y in macro_board)
    max_y = max(y for _, y in macro_board)
    bbox_w = max_x - min_x + 1
    bbox_h = max_y - min_y + 1
    bbox_area = bbox_w * bbox_h
    fill_ratio = len(macro_board) / bbox_area
    aspect = max(bbox_w, bbox_h) / max(1, min(bbox_w, bbox_h))
    perimeter = 0
    for cell in board:
        perimeter += sum(1 for neighbor in neighbors4(cell) if neighbor not in board)
    ideal_perimeter = 2 * (math.sqrt(len(board)) + math.sqrt(len(board)))
    score += fill_ratio * 900.0
    score -= max(0.0, aspect - 1.8) * 160.0
    score -= max(0.0, perimeter - ideal_perimeter) * 1.7

    signatures = [canonical_signature(piece) for piece in pieces]
    duplicate_penalty = len(signatures) - len(set(signatures))
    score -= duplicate_penalty * 60.0
    skinny_penalty = 0
    for piece in pieces:
        min_px, min_py, max_px, max_py = bounds(piece)
        pw = max_px - min_px + 1
        ph = max_py - min_py + 1
        if max(pw, ph) >= 10 and min(pw, ph) <= 2:
            skinny_penalty += 1
    score -= skinny_penalty * 35.0
    return score


def generate_pair_swap_candidate(
    rng: random.Random,
    args: argparse.Namespace,
    attempt_index: int,
) -> Candidate | None:
    """Generate a chunky fixed-orientation multi-solution candidate.

    The idea is simple and very useful for physical tests: choose four robust
    shapes, use two copies of each shape, then pack those eight pieces into one
    connected board.  With piece artwork/IDs present, the player still has to
    decide which same-outline piece goes to which same-outline slot, and the
    fixed-orientation solver verifies that multiple complete assignments exist.
    """
    if not args.allow_identical_pieces:
        return None
    if args.pieces % 2 != 0 or args.pieces < 2:
        return None
    library = robust_pair_shape_library()
    pair_count = args.pieces // 2
    if len(library) < pair_count:
        return None

    board_w_small = args.board_w * SCALE
    board_h_small = args.board_h * SCALE
    base_indices = rng.sample(range(len(library)), pair_count)
    piece_indices = [index for index in base_indices for _ in range(2)]
    rng.shuffle(piece_indices)

    occupied: set[Cell] = set()
    for placed_count, shape_index in enumerate(piece_indices):
        shape = library[shape_index]
        _, _, max_shape_x, max_shape_y = bounds(shape)
        options: list[tuple[float, set[Cell]]] = []
        for ty in range(0, board_h_small - max_shape_y, SCALE):
            for tx in range(0, board_w_small - max_shape_x, SCALE):
                placed = {(x + tx, y + ty) for x, y in shape}
                if occupied & placed:
                    continue
                new_board = occupied | placed
                if not is_legal_half_cell_shape(new_board):
                    continue
                if placed_count > 0 and not any(
                    neighbor in occupied for cell in placed for neighbor in neighbors4(cell)
                ):
                    continue
                min_x, min_y, max_x, max_y = bounds(new_board)
                bbox_w = max_x - min_x + 1
                bbox_h = max_y - min_y + 1
                bbox_area = bbox_w * bbox_h
                fill_ratio = len(new_board) / bbox_area
                aspect = max(bbox_w, bbox_h) / max(1, min(bbox_w, bbox_h))
                aspect_penalty = abs(aspect - 1.35)
                touch = sum(
                    1 for cell in placed for neighbor in neighbors4(cell) if neighbor in occupied
                )
                center_x = sum(x for x, _ in placed) / len(placed)
                center_y = sum(y for _, y in placed) / len(placed)
                center_bias = abs(center_x - board_w_small / 2) + abs(center_y - board_h_small / 2)
                score = (
                    bbox_area * 2.5
                    - fill_ratio * 45.0
                    + aspect_penalty * 18.0
                    - touch * 1.5
                    + center_bias * 0.02
                    + rng.random() * 0.2
                )
                options.append((score, placed))
        if not options:
            return None
        options.sort(key=lambda item: item[0])
        chosen = options[0][1]
        occupied |= chosen

    board = normalize_cells(occupied)
    pieces = [normalize_cells(library[index]) for index in piece_indices]
    board, pieces = prefer_landscape(board, pieces)
    if len(board) != args.pieces * PIECE_AREA_SMALL:
        return None
    if not is_connected(board):
        return None
    if not is_legal_half_cell_shape(board):
        return None
    if not args.allow_holes and has_small_hole(board):
        return None
    if has_quarter_artifact(board):
        return None
    if any(not validate_piece(piece, min_half_cells=args.min_half_cells) for piece in pieces):
        return None

    solution_count, solutions, placements_by_piece = count_solutions(
        board=board,
        pieces=pieces,
        allow_rotate=False,
        allow_mirror=False,
        limit=args.solution_count_limit,
    )
    if not (args.min_solutions <= solution_count <= args.max_solutions):
        return None

    rotated_solution_count, _, _ = count_solutions(
        board=board,
        pieces=pieces,
        allow_rotate=not args.no_rotate,
        allow_mirror=args.allow_mirror,
        limit=args.solution_count_limit,
    )
    analysis = analyze_candidate(
        board,
        pieces,
        solutions,
        solution_count,
        rotated_solution_count,
        placements_by_piece,
        args.min_solutions,
        args.max_solutions,
    )
    if analysis.quarter_artifact_count != 0 or analysis.fragile_artifact_count != 0:
        return None
    if analysis.duplicate_piece_count != 0 and not args.allow_identical_pieces:
        return None
    return Candidate(
        board=board,
        pieces=pieces,
        solutions=solutions,
        solution_count=solution_count,
        placements_by_piece=placements_by_piece,
        score=analysis.difficulty_score,
        analysis=analysis,
        attempts=attempt_index,
    )


def generate_candidate(
    rng: random.Random,
    args: argparse.Namespace,
    attempt_index: int,
) -> Candidate | None:
    macro_area = args.pieces * (PIECE_AREA_SMALL // 4)
    macro_board = random_macro_board(
        rng,
        width=args.board_w,
        height=args.board_h,
        area=macro_area,
        allow_holes=args.allow_holes,
        max_tries=80,
    )
    if macro_board is None:
        return None
    regions = partition_into_tetrominoes(macro_board, args.pieces, rng)
    if regions is None:
        return None
    pieces = make_swapped_pieces(regions, rng, min_half_cells=args.min_half_cells)
    if pieces is None:
        return None
    board = macro_to_full_small(macro_board)
    board, pieces = normalize_board_and_pieces(board, pieces)
    board, pieces = prefer_landscape(board, pieces)

    if len(board) != args.pieces * PIECE_AREA_SMALL:
        return None
    if not is_connected(board):
        return None
    if not is_legal_half_cell_shape(board):
        return None
    if not args.allow_holes and has_small_hole(board):
        return None
    if has_quarter_artifact(board):
        return None
    if any(has_quarter_artifact(piece) for piece in pieces):
        return None
    if any(not validate_piece(piece, min_half_cells=args.min_half_cells) for piece in pieces):
        return None

    solution_count, solutions, placements_by_piece = count_solutions(
        board=board,
        pieces=pieces,
        allow_rotate=False,
        allow_mirror=False,
        limit=args.solution_count_limit,
    )
    rotated_solution_count, _, _ = count_solutions(
        board=board,
        pieces=pieces,
        allow_rotate=not args.no_rotate,
        allow_mirror=args.allow_mirror,
        limit=args.solution_count_limit,
    )
    if not (args.min_solutions <= solution_count <= args.max_solutions):
        return None
    analysis = analyze_candidate(
        board,
        pieces,
        solutions,
        solution_count,
        rotated_solution_count,
        placements_by_piece,
        args.min_solutions,
        args.max_solutions,
    )
    if analysis.quarter_artifact_count != 0 or analysis.fragile_artifact_count != 0:
        return None
    if analysis.duplicate_piece_count != 0 and not args.allow_identical_pieces:
        return None
    return Candidate(
        board=board,
        pieces=pieces,
        solutions=solutions,
        solution_count=solution_count,
        placements_by_piece=placements_by_piece,
        score=analysis.difficulty_score,
        analysis=analysis,
        attempts=attempt_index,
    )


def guided_regions_from_super_layout(super_layout: list[MacroCell]) -> list[set[MacroCell]]:
    regions = []
    for sx, sy in super_layout:
        bx, by = sx * 2, sy * 2
        regions.append({(bx + dx, by + dy) for dx in range(2) for dy in range(2)})
    return regions


def generate_guided_candidate(
    args: argparse.Namespace,
    layout_index: int,
    swap_seed: int,
    attempt_index: int,
) -> Candidate | None:
    if args.pieces != 8:
        return None
    super_layout = GUIDED_SUPER_LAYOUTS[layout_index]
    max_super_x = max(x for x, _ in super_layout)
    max_super_y = max(y for _, y in super_layout)
    if (max_super_x + 1) * 2 > args.board_w or (max_super_y + 1) * 2 > args.board_h:
        return None

    regions = guided_regions_from_super_layout(super_layout)
    macro_board = {cell for region in regions for cell in region}
    if not args.allow_holes and has_macro_hole(macro_board):
        return None

    pieces = make_swapped_pieces(
        regions,
        random.Random(swap_seed),
        min_half_cells=args.min_half_cells,
        max_tries=1,
    )
    if pieces is None:
        return None

    signatures = [canonical_signature(piece) for piece in pieces]
    if len(set(signatures)) != len(signatures):
        return None

    board = macro_to_full_small(macro_board)
    board, pieces = normalize_board_and_pieces(board, pieces)
    board, pieces = prefer_landscape(board, pieces)
    if not is_connected(board):
        return None
    if not is_legal_half_cell_shape(board):
        return None
    if not args.allow_holes and has_small_hole(board):
        return None
    if any(not validate_piece(piece, min_half_cells=args.min_half_cells) for piece in pieces):
        return None

    solution_count, solutions, placements_by_piece = count_solutions(
        board=board,
        pieces=pieces,
        allow_rotate=False,
        allow_mirror=False,
        limit=args.solution_count_limit,
    )
    rotated_solution_count, _, _ = count_solutions(
        board=board,
        pieces=pieces,
        allow_rotate=not args.no_rotate,
        allow_mirror=args.allow_mirror,
        limit=args.solution_count_limit,
    )
    if not (args.min_solutions <= solution_count <= args.max_solutions):
        return None
    analysis = analyze_candidate(
        board,
        pieces,
        solutions,
        solution_count,
        rotated_solution_count,
        placements_by_piece,
        args.min_solutions,
        args.max_solutions,
    )
    if analysis.quarter_artifact_count != 0 or analysis.fragile_artifact_count != 0:
        return None
    if analysis.duplicate_piece_count != 0 and not args.allow_identical_pieces:
        return None
    return Candidate(
        board=board,
        pieces=pieces,
        solutions=solutions,
        solution_count=solution_count,
        placements_by_piece=placements_by_piece,
        score=analysis.difficulty_score,
        analysis=analysis,
        attempts=attempt_index,
    )


def generate_guided_candidates(
    args: argparse.Namespace,
    start_time: float,
    max_count: int,
) -> list[Candidate]:
    candidates: list[Candidate] = []
    if args.pieces != 8:
        return candidates

    # Seed 982 on layout 0 is a known good non-rectangular, non-duplicate
    # generated case for the default constraints.  The rest keep this as a
    # small guided search rather than a single hard-coded puzzle.
    seed_order = [982, 237, 124, 0, 1, 2, 3, 5, 8, 13, 21]
    known_seeds = set(seed_order)
    seed_order.extend(seed for seed in range(4_000) if seed not in known_seeds)
    attempt = 0
    seen_json: set[str] = set()
    for layout_index in range(len(GUIDED_SUPER_LAYOUTS)):
        for swap_seed in seed_order:
            if len(candidates) >= max_count:
                return candidates
            if time.monotonic() - start_time > args.time_limit:
                return candidates
            attempt += 1
            candidate = generate_guided_candidate(args, layout_index, swap_seed, attempt)
            if candidate is None:
                continue
            key = json.dumps(
                {
                    "board": sorted(candidate.board),
                    "pieces": [sorted(piece) for piece in candidate.pieces],
                }
            )
            if key in seen_json:
                continue
            seen_json.add(key)
            candidates.append(candidate)
            candidates.sort(key=lambda c: c.score, reverse=True)
            if args.verbose:
                print(
                    f"guided accepted #{len(candidates)} layout={layout_index} seed={swap_seed}: "
                    f"solutions={candidate.solution_count}, score={candidate.score:.1f}",
                    file=sys.stderr,
                )
    return candidates


def generate_candidates(args: argparse.Namespace) -> list[Candidate]:
    rng = random.Random(args.seed)
    start = time.monotonic()
    candidates: list[Candidate] = []
    seen_json: set[str] = set()
    attempts = 0
    pair_attempt_limit = max(args.candidates * 60, 360) if args.allow_identical_pieces else 0
    while attempts < pair_attempt_limit:
        attempts += 1
        if time.monotonic() - start > args.time_limit:
            break
        candidate = generate_pair_swap_candidate(rng, args, attempts)
        if candidate is None:
            if args.verbose and attempts % 25 == 0:
                print(f"pair attempts={attempts}, candidates={len(candidates)}", file=sys.stderr)
            continue
        key = json.dumps(
            {
                "board": sorted(candidate.board),
                "pieces": [sorted(piece) for piece in candidate.pieces],
            }
        )
        if key in seen_json:
            continue
        seen_json.add(key)
        candidates.append(candidate)
        candidates.sort(key=lambda c: c.score, reverse=True)
        if args.verbose:
            print(
                f"pair accepted #{len(candidates)} at attempt {attempts}: "
                f"fixed_solutions={candidate.solution_count}, score={candidate.score:.1f}",
                file=sys.stderr,
            )

    if len(candidates) < args.candidates and time.monotonic() - start <= args.time_limit:
        guided = generate_guided_candidates(args, start, args.candidates - len(candidates))
        for candidate in guided:
            key = json.dumps(
                {
                    "board": sorted(candidate.board),
                    "pieces": [sorted(piece) for piece in candidate.pieces],
                }
            )
            if key in seen_json:
                continue
            seen_json.add(key)
            candidates.append(candidate)
        candidates.sort(key=lambda c: c.score, reverse=True)

    while len(candidates) < args.candidates:
        attempts += 1
        if time.monotonic() - start > args.time_limit:
            break
        candidate = generate_candidate(rng, args, attempts)
        if candidate is None:
            if args.verbose and attempts % 25 == 0:
                print(f"attempts={attempts}, candidates={len(candidates)}", file=sys.stderr)
            continue
        key = json.dumps(
            {
                "board": sorted(candidate.board),
                "pieces": [sorted(piece) for piece in candidate.pieces],
            }
        )
        if key in seen_json:
            continue
        seen_json.add(key)
        candidates.append(candidate)
        candidates.sort(key=lambda c: c.score, reverse=True)
        if args.verbose:
            print(
                f"accepted #{len(candidates)} at attempt {attempts}: "
                f"solutions={candidate.solution_count}, score={candidate.score:.1f}",
                file=sys.stderr,
            )
    return candidates[: args.candidates]


def grid_string(
    cells: set[Cell] | frozenset[Cell],
    filled: str = "#",
    empty: str = ".",
    pad_to: tuple[int, int] | None = None,
) -> str:
    if not cells:
        return ""
    min_x, min_y, max_x, max_y = bounds(cells)
    if pad_to is not None:
        max_x = max(max_x, pad_to[0] - 1)
        max_y = max(max_y, pad_to[1] - 1)
        min_x = min(min_x, 0)
        min_y = min(min_y, 0)
    rows = []
    for y in range(min_y, max_y + 1):
        rows.append("".join(filled if (x, y) in cells else empty for x in range(min_x, max_x + 1)))
    return "\n".join(rows)


def solution_grid_string(board: set[Cell], solution: dict[int, frozenset[Cell]]) -> str:
    min_x, min_y, max_x, max_y = bounds(board)
    lookup: dict[Cell, str] = {}
    for piece_index, cells in solution.items():
        letter = LETTERS[piece_index % len(LETTERS)]
        for cell in cells:
            lookup[cell] = letter
    rows = []
    for y in range(min_y, max_y + 1):
        row = []
        for x in range(min_x, max_x + 1):
            if (x, y) not in board:
                row.append(".")
            else:
                row.append(lookup.get((x, y), "?"))
        rows.append("".join(row))
    return "\n".join(rows)


def candidate_to_text(candidate: Candidate, max_solutions: int = 2) -> str:
    lines = []
    lines.append("Board:")
    lines.append(grid_string(candidate.board))
    lines.append("")
    for i, solution in enumerate(candidate.solutions[:max_solutions], start=1):
        lines.append(f"Solution {i}:")
        lines.append(solution_grid_string(candidate.board, solution))
        lines.append("")
    for i, piece in enumerate(candidate.pieces):
        lines.append(f"Piece {LETTERS[i]}:")
        lines.append(grid_string(normalize_cells(piece)))
        lines.append("")
    lines.append("Analysis:")
    lines.append(json.dumps(analysis_to_json(candidate.analysis), ensure_ascii=False, indent=2))
    return "\n".join(lines)


def analysis_to_json(analysis: Analysis) -> dict[str, object]:
    return {
        "solution_count": analysis.solution_count,
        "rotated_solution_count": analysis.rotated_solution_count,
        "fixed_pieces": analysis.fixed_pieces,
        "movable_pieces": analysis.movable_pieces,
        "average_piece_candidates": analysis.average_piece_candidates,
        "half_cell_count_per_piece": analysis.half_cell_count_per_piece,
        "total_half_cell_count": analysis.total_half_cell_count,
        "quarter_artifact_count": analysis.quarter_artifact_count,
        "fragile_artifact_count": analysis.fragile_artifact_count,
        "duplicate_piece_count": analysis.duplicate_piece_count,
        "difficulty_score": analysis.difficulty_score,
    }


def candidate_to_json(candidate: Candidate) -> dict[str, object]:
    return {
        "scale": SCALE,
        "piece_count": len(candidate.pieces),
        "board": sorted([list(cell) for cell in candidate.board]),
        "pieces": [
            {
                "id": LETTERS[i],
                "cells": sorted([list(cell) for cell in normalize_cells(piece)]),
                "half_cell_count": count_half_cells(piece),
            }
            for i, piece in enumerate(candidate.pieces)
        ],
        "solution_count": candidate.solution_count,
        "rotated_solution_count": candidate.analysis.rotated_solution_count,
        "solutions": [
            {
                LETTERS[piece_index]: sorted([list(cell) for cell in cells])
                for piece_index, cells in sorted(solution.items())
            }
            for solution in candidate.solutions
        ],
        "score": candidate.score,
        "quarter_artifact_count": candidate.analysis.quarter_artifact_count,
        "fragile_artifact_count": candidate.analysis.fragile_artifact_count,
        "duplicate_piece_count": candidate.analysis.duplicate_piece_count,
        "analysis": analysis_to_json(candidate.analysis),
    }


def write_outputs(candidates: list[Candidate], output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    index_path = output_dir / "candidates.json"
    with index_path.open("w", encoding="utf-8") as f:
        json.dump([candidate_to_json(c) for c in candidates], f, ensure_ascii=False, indent=2)
    write_gallery_html(candidates, output_dir / "index.html")

    if not candidates:
        return

    for idx, candidate in enumerate(candidates, start=1):
        prefix = f"candidate_{idx:03d}"
        with (output_dir / f"{prefix}.txt").open("w", encoding="utf-8") as f:
            f.write(candidate_to_text(candidate))
        with (output_dir / f"{prefix}.json").open("w", encoding="utf-8") as f:
            json.dump(candidate_to_json(candidate), f, ensure_ascii=False, indent=2)
        write_candidate_svgs(candidate, output_dir, prefix)

    # Required convenient names for the best candidate.
    best = candidates[0]
    with (output_dir / "candidate.txt").open("w", encoding="utf-8") as f:
        f.write(candidate_to_text(best))
    with (output_dir / "candidate.json").open("w", encoding="utf-8") as f:
        json.dump(candidate_to_json(best), f, ensure_ascii=False, indent=2)
    write_candidate_svgs(best, output_dir, "candidate")


def write_gallery_html(candidates: list[Candidate], path: Path) -> None:
    data = json.dumps([candidate_to_json(c) for c in candidates], ensure_ascii=False)
    html = f"""<!doctype html>
<html lang="ja">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Half-cell puzzle gallery</title>
  <style>
    :root {{
      --bg: #f6f4ee;
      --ink: #202124;
      --muted: #5f6368;
      --line: #d4cfc4;
      --panel: #ffffff;
      --accent: #0f766e;
      --warn: #b45309;
    }}
    * {{ box-sizing: border-box; }}
    body {{
      margin: 0;
      font-family: Arial, "Hiragino Kaku Gothic ProN", "Yu Gothic", Meiryo, sans-serif;
      color: var(--ink);
      background: var(--bg);
    }}
    header {{
      display: flex;
      align-items: end;
      justify-content: space-between;
      gap: 16px;
      padding: 18px 22px 14px;
      border-bottom: 1px solid var(--line);
      background: #fffdf8;
    }}
    h1 {{
      margin: 0;
      font-size: 22px;
      letter-spacing: 0;
    }}
    .summary {{
      display: flex;
      gap: 10px;
      flex-wrap: wrap;
      justify-content: flex-end;
      color: var(--muted);
      font-size: 13px;
    }}
    .summary span, .metric {{
      border: 1px solid var(--line);
      background: #fff;
      border-radius: 6px;
      padding: 5px 8px;
      white-space: nowrap;
    }}
    main {{
      display: grid;
      grid-template-columns: minmax(260px, 34vw) 1fr;
      min-height: calc(100vh - 68px);
    }}
    #cards {{
      padding: 16px;
      overflow: auto;
      border-right: 1px solid var(--line);
      display: grid;
      align-content: start;
      gap: 12px;
    }}
    .card {{
      border: 1px solid var(--line);
      background: var(--panel);
      border-radius: 8px;
      padding: 10px;
      cursor: pointer;
      display: grid;
      gap: 8px;
    }}
    .card.active {{
      outline: 2px solid var(--accent);
      outline-offset: 1px;
    }}
    .card-top {{
      display: flex;
      align-items: center;
      justify-content: space-between;
      gap: 8px;
      font-size: 13px;
      color: var(--muted);
    }}
    .card-title {{
      font-size: 15px;
      color: var(--ink);
      font-weight: 700;
    }}
    .mini {{
      width: 100%;
      max-height: 150px;
      display: block;
      border: 1px solid var(--line);
      background: #faf9f5;
    }}
    #detail {{
      padding: 18px;
      overflow: auto;
      display: grid;
      gap: 16px;
      align-content: start;
    }}
    .detail-head {{
      display: flex;
      align-items: center;
      justify-content: space-between;
      gap: 14px;
      flex-wrap: wrap;
    }}
    .detail-head h2 {{
      margin: 0;
      font-size: 20px;
      letter-spacing: 0;
    }}
    .metrics {{
      display: flex;
      flex-wrap: wrap;
      gap: 8px;
      font-size: 13px;
      color: var(--muted);
    }}
    .section {{
      display: grid;
      gap: 8px;
    }}
    .section h3 {{
      margin: 0;
      font-size: 15px;
      letter-spacing: 0;
    }}
    .viewer {{
      width: 100%;
      overflow: auto;
      border: 1px solid var(--line);
      background: #fffdf8;
      border-radius: 8px;
      padding: 10px;
    }}
    .solution-grid {{
      display: grid;
      grid-template-columns: repeat(auto-fit, minmax(280px, 1fr));
      gap: 12px;
    }}
    .pieces {{
      display: grid;
      grid-template-columns: repeat(auto-fill, minmax(140px, 1fr));
      gap: 10px;
    }}
    .piece {{
      border: 1px solid var(--line);
      border-radius: 8px;
      background: #fff;
      padding: 8px;
    }}
    .piece-title {{
      font-size: 13px;
      font-weight: 700;
      margin-bottom: 6px;
    }}
    svg {{ max-width: 100%; height: auto; display: block; }}
    @media (max-width: 860px) {{
      header {{ align-items: start; flex-direction: column; }}
      main {{ grid-template-columns: 1fr; }}
      #cards {{
        border-right: 0;
        border-bottom: 1px solid var(--line);
        grid-template-columns: repeat(auto-fill, minmax(240px, 1fr));
        max-height: 48vh;
      }}
    }}
  </style>
</head>
<body>
  <header>
    <h1>半マスずれポリオミノ候補</h1>
    <div class="summary" id="summary"></div>
  </header>
  <main>
    <section id="cards"></section>
    <section id="detail"></section>
  </main>
  <script type="application/json" id="candidate-data">{data}</script>
  <script>
    const DATA = JSON.parse(document.getElementById('candidate-data').textContent);
    const COLORS = {json.dumps(COLORS)};
    let selected = 0;

    function cellBounds(cells) {{
      let xs = cells.map(c => c[0]);
      let ys = cells.map(c => c[1]);
      return [Math.min(...xs), Math.min(...ys), Math.max(...xs), Math.max(...ys)];
    }}

    function key(c) {{ return c[0] + ',' + c[1]; }}

    function svgGrid(cells, assignment, labels, scale) {{
      if (!cells.length) return '';
      const [minX, minY, maxX, maxY] = cellBounds(cells);
      const size = scale || 14;
      const pad = 10;
      const w = (maxX - minX + 1) * size + pad * 2;
      const h = (maxY - minY + 1) * size + pad * 2;
      const cellSet = new Set(cells.map(key));
      let rects = '';
      for (const c of cells) {{
        const k = key(c);
        const p = assignment ? assignment[k] : -1;
        const fill = p >= 0 ? COLORS[p % COLORS.length] : '#cfd8dc';
        const x = pad + (c[0] - minX) * size;
        const y = pad + (c[1] - minY) * size;
        rects += `<rect x="${{x}}" y="${{y}}" width="${{size}}" height="${{size}}" fill="${{fill}}"/>`;
      }}
      let lines = '';
      for (let x = minX; x <= maxX + 1; x++) {{
        const sx = pad + (x - minX) * size;
        const strong = x % 2 === 0;
        lines += `<line x1="${{sx}}" y1="${{pad}}" x2="${{sx}}" y2="${{pad + (maxY - minY + 1) * size}}" stroke="${{strong ? '#817c70' : '#d7d2c6'}}" stroke-width="${{strong ? 1.4 : 0.7}}"/>`;
      }}
      for (let y = minY; y <= maxY + 1; y++) {{
        const sy = pad + (y - minY) * size;
        const strong = y % 2 === 0;
        lines += `<line x1="${{pad}}" y1="${{sy}}" x2="${{pad + (maxX - minX + 1) * size}}" y2="${{sy}}" stroke="${{strong ? '#817c70' : '#d7d2c6'}}" stroke-width="${{strong ? 1.4 : 0.7}}"/>`;
      }}
      let borders = '';
      if (assignment) {{
        const dirs = [[-1,0,'l'], [1,0,'r'], [0,-1,'t'], [0,1,'b']];
        for (const c of cells) {{
          const p = assignment[key(c)];
          const x = pad + (c[0] - minX) * size;
          const y = pad + (c[1] - minY) * size;
          for (const [dx, dy, side] of dirs) {{
            const nk = (c[0] + dx) + ',' + (c[1] + dy);
            if (cellSet.has(nk) && assignment[nk] === p) continue;
            if (side === 'l') borders += `<line x1="${{x}}" y1="${{y}}" x2="${{x}}" y2="${{y + size}}" stroke="#222" stroke-width="2"/>`;
            if (side === 'r') borders += `<line x1="${{x + size}}" y1="${{y}}" x2="${{x + size}}" y2="${{y + size}}" stroke="#222" stroke-width="2"/>`;
            if (side === 't') borders += `<line x1="${{x}}" y1="${{y}}" x2="${{x + size}}" y2="${{y}}" stroke="#222" stroke-width="2"/>`;
            if (side === 'b') borders += `<line x1="${{x}}" y1="${{y + size}}" x2="${{x + size}}" y2="${{y + size}}" stroke="#222" stroke-width="2"/>`;
          }}
        }}
      }}
      let text = '';
      if (labels && assignment) {{
        for (const label of Object.keys(labels)) {{
          const pts = labels[label];
          const cx = pts.reduce((a, c) => a + c[0], 0) / pts.length;
          const cy = pts.reduce((a, c) => a + c[1], 0) / pts.length;
          const sx = pad + (cx - minX + 0.5) * size;
          const sy = pad + (cy - minY + 0.5) * size;
          text += `<text x="${{sx}}" y="${{sy}}" text-anchor="middle" dominant-baseline="central" font-size="${{Math.max(10, size)}}" font-weight="700" font-family="Arial" fill="#202020" paint-order="stroke" stroke="#fff" stroke-width="3">${{label}}</text>`;
        }}
      }}
      return `<svg viewBox="0 0 ${{w}} ${{h}}" role="img" aria-label="candidate grid">${{rects}}${{borders}}${{lines}}${{text}}</svg>`;
    }}

    function assignmentFromSolution(solution) {{
      const assignment = {{}};
      const labels = {{}};
      Object.keys(solution).forEach((id, idx) => {{
        labels[id] = solution[id];
        for (const c of solution[id]) assignment[key(c)] = idx;
      }});
      return [assignment, labels];
    }}

    function renderCards() {{
      const cards = document.getElementById('cards');
      cards.innerHTML = DATA.map((c, i) => `
        <article class="card ${{i === selected ? 'active' : ''}}" data-i="${{i}}">
          <div class="card-top"><span class="card-title">候補 ${{i + 1}}</span><span>固定 ${{c.solution_count}} 解</span></div>
          <div class="mini">${{svgGrid(c.board, null, null, 7)}}</div>
          <div class="metrics">
            <span class="metric">脆さ ${{c.fragile_artifact_count}}</span>
            <span class="metric">重複 ${{c.duplicate_piece_count}}</span>
            <span class="metric">1/4 ${{c.quarter_artifact_count}}</span>
            <span class="metric">半マス ${{c.analysis.total_half_cell_count}}</span>
          </div>
        </article>
      `).join('');
      cards.querySelectorAll('.card').forEach(card => {{
        card.addEventListener('click', () => {{
          selected = Number(card.dataset.i);
          render();
        }});
      }});
    }}

    function renderDetail() {{
      const c = DATA[selected];
      const detail = document.getElementById('detail');
      const solutions = c.solutions.slice(0, 8).map((s, i) => {{
        const [a, labels] = assignmentFromSolution(s);
        return `<div class="viewer"><h3>Solution ${{i + 1}}</h3>${{svgGrid(c.board, a, labels, 16)}}</div>`;
      }}).join('');
      const pieces = c.pieces.map((p, i) => `
        <div class="piece">
          <div class="piece-title">Piece ${{p.id}} / 半マス ${{p.half_cell_count}}</div>
          ${{svgGrid(p.cells, Object.fromEntries(p.cells.map(cell => [key(cell), i])), null, 14)}}
        </div>
      `).join('');
      detail.innerHTML = `
        <div class="detail-head">
          <h2>候補 ${{selected + 1}}</h2>
          <div class="metrics">
            <span class="metric">固定向き ${{c.solution_count}} 解</span>
            <span class="metric">回転あり ${{c.rotated_solution_count}} 解</span>
            <span class="metric">脆さ ${{c.fragile_artifact_count}}</span>
            <span class="metric">重複 ${{c.duplicate_piece_count}}</span>
            <span class="metric">1/4 ${{c.quarter_artifact_count}}</span>
            <span class="metric">Score ${{c.score.toFixed(1)}}</span>
          </div>
        </div>
        <div class="section">
          <h3>Board</h3>
          <div class="viewer">${{svgGrid(c.board, null, null, 16)}}</div>
        </div>
        <div class="section">
          <h3>Solutions</h3>
          <div class="solution-grid">${{solutions}}</div>
        </div>
        <div class="section">
          <h3>Pieces</h3>
          <div class="pieces">${{pieces}}</div>
        </div>
      `;
    }}

    function renderSummary() {{
      document.getElementById('summary').innerHTML = `
        <span>${{DATA.length}} candidates</span>
        <span>fixed orientation</span>
        <span>fragile = 0</span>
        <span>duplicate = 0</span>
        <span>quarter = 0</span>
      `;
    }}

    function render() {{
      renderSummary();
      renderCards();
      renderDetail();
    }}
    render();
  </script>
</body>
</html>
"""
    path.write_text(html, encoding="utf-8")


def write_candidate_svgs(candidate: Candidate, output_dir: Path, prefix: str) -> None:
    (output_dir / f"{prefix}_board.svg").write_text(
        svg_board(candidate.board), encoding="utf-8"
    )
    (output_dir / f"{prefix}_pieces.svg").write_text(
        svg_pieces(candidate.pieces), encoding="utf-8"
    )
    for i, solution in enumerate(candidate.solutions[:2], start=1):
        (output_dir / f"{prefix}_solution_{i}.svg").write_text(
            svg_solution(candidate.board, solution), encoding="utf-8"
        )


def svg_header(width: int, height: int) -> str:
    return (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" '
        f'viewBox="0 0 {width} {height}">\n'
        '<rect width="100%" height="100%" fill="#f8f7f2"/>\n'
    )


def svg_grid(
    min_x: int,
    min_y: int,
    max_x: int,
    max_y: int,
    cell_size: int,
    ox: int,
    oy: int,
) -> str:
    lines = []
    width = (max_x - min_x + 1) * cell_size
    height = (max_y - min_y + 1) * cell_size
    for x in range(min_x, max_x + 2):
        sx = ox + (x - min_x) * cell_size
        stroke = "#817c70" if x % SCALE == 0 else "#d7d2c6"
        sw = 1.6 if x % SCALE == 0 else 0.8
        lines.append(
            f'<line x1="{sx}" y1="{oy}" x2="{sx}" y2="{oy + height}" '
            f'stroke="{stroke}" stroke-width="{sw}"/>'
        )
    for y in range(min_y, max_y + 2):
        sy = oy + (y - min_y) * cell_size
        stroke = "#817c70" if y % SCALE == 0 else "#d7d2c6"
        sw = 1.6 if y % SCALE == 0 else 0.8
        lines.append(
            f'<line x1="{ox}" y1="{sy}" x2="{ox + width}" y2="{sy}" '
            f'stroke="{stroke}" stroke-width="{sw}"/>'
        )
    return "\n".join(lines)


def svg_board(board: set[Cell]) -> str:
    min_x, min_y, max_x, max_y = bounds(board)
    cell_size = 18
    margin = 28
    width = (max_x - min_x + 1) * cell_size + margin * 2
    height = (max_y - min_y + 1) * cell_size + margin * 2
    parts = [svg_header(width, height)]
    for x, y in sorted(board):
        sx = margin + (x - min_x) * cell_size
        sy = margin + (y - min_y) * cell_size
        parts.append(
            f'<rect x="{sx}" y="{sy}" width="{cell_size}" height="{cell_size}" '
            'fill="#cfd8dc" stroke="none"/>'
        )
    parts.append(svg_grid(min_x, min_y, max_x, max_y, cell_size, margin, margin))
    parts.append("</svg>\n")
    return "\n".join(parts)


def svg_solution(board: set[Cell], solution: dict[int, frozenset[Cell]]) -> str:
    min_x, min_y, max_x, max_y = bounds(board)
    cell_size = 18
    margin = 28
    width = (max_x - min_x + 1) * cell_size + margin * 2
    height = (max_y - min_y + 1) * cell_size + margin * 2
    lookup: dict[Cell, int] = {}
    for piece_index, cells in solution.items():
        for cell in cells:
            lookup[cell] = piece_index

    parts = [svg_header(width, height)]
    for cell, piece_index in sorted(lookup.items()):
        x, y = cell
        sx = margin + (x - min_x) * cell_size
        sy = margin + (y - min_y) * cell_size
        color = COLORS[piece_index % len(COLORS)]
        parts.append(
            f'<rect x="{sx}" y="{sy}" width="{cell_size}" height="{cell_size}" '
            f'fill="{color}" stroke="none"/>'
        )

    # Thick boundaries where adjacent small cells belong to different pieces.
    for x, y in sorted(board):
        piece_index = lookup.get((x, y))
        sx = margin + (x - min_x) * cell_size
        sy = margin + (y - min_y) * cell_size
        for edge, neighbor in (
            ("left", (x - 1, y)),
            ("right", (x + 1, y)),
            ("top", (x, y - 1)),
            ("bottom", (x, y + 1)),
        ):
            if neighbor in board and lookup.get(neighbor) == piece_index:
                continue
            if edge == "left":
                parts.append(boundary_line(sx, sy, sx, sy + cell_size))
            elif edge == "right":
                parts.append(boundary_line(sx + cell_size, sy, sx + cell_size, sy + cell_size))
            elif edge == "top":
                parts.append(boundary_line(sx, sy, sx + cell_size, sy))
            else:
                parts.append(boundary_line(sx, sy + cell_size, sx + cell_size, sy + cell_size))

    parts.append(svg_grid(min_x, min_y, max_x, max_y, cell_size, margin, margin))
    for piece_index, cells in solution.items():
        cx = sum(x for x, _ in cells) / len(cells)
        cy = sum(y for _, y in cells) / len(cells)
        sx = margin + (cx - min_x + 0.5) * cell_size
        sy = margin + (cy - min_y + 0.5) * cell_size
        parts.append(
            f'<text x="{sx:.1f}" y="{sy:.1f}" text-anchor="middle" dominant-baseline="central" '
            'font-family="Arial, sans-serif" font-size="16" font-weight="700" '
            'fill="#202020" paint-order="stroke" stroke="#ffffff" stroke-width="3">'
            f"{LETTERS[piece_index]}</text>"
        )
    parts.append("</svg>\n")
    return "\n".join(parts)


def boundary_line(x1: float, y1: float, x2: float, y2: float) -> str:
    return (
        f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" '
        'stroke="#1f1f1f" stroke-width="2.3" stroke-linecap="square"/>'
    )


def svg_pieces(pieces: list[set[Cell]]) -> str:
    cell_size = 18
    margin = 28
    gap_x = 42
    gap_y = 48
    panels = []
    panel_widths = []
    panel_heights = []
    for i, piece in enumerate(pieces):
        piece = normalize_cells(piece)
        min_x, min_y, max_x, max_y = bounds(piece)
        w = (max_x - min_x + 1) * cell_size
        h = (max_y - min_y + 1) * cell_size
        panel_widths.append(max(w, 72))
        panel_heights.append(h + 22)
        panels.append((i, piece, min_x, min_y, max_x, max_y, w, h))

    cols = min(4, len(pieces))
    rows = math.ceil(len(pieces) / cols)
    col_width = max(panel_widths) + gap_x
    row_height = max(panel_heights) + gap_y
    width = margin * 2 + cols * col_width - gap_x
    height = margin * 2 + rows * row_height - gap_y
    parts = [svg_header(width, height)]

    for i, piece, min_x, min_y, max_x, max_y, _, _ in panels:
        col = i % cols
        row = i // cols
        ox = margin + col * col_width
        oy = margin + row * row_height + 22
        parts.append(
            f'<text x="{ox}" y="{oy - 8}" font-family="Arial, sans-serif" '
            'font-size="15" font-weight="700" fill="#202020">'
            f"Piece {LETTERS[i]}</text>"
        )
        for x, y in sorted(piece):
            sx = ox + (x - min_x) * cell_size
            sy = oy + (y - min_y) * cell_size
            color = COLORS[i % len(COLORS)]
            parts.append(
                f'<rect x="{sx}" y="{sy}" width="{cell_size}" height="{cell_size}" '
                f'fill="{color}" stroke="none"/>'
            )
        for x, y in sorted(piece):
            sx = ox + (x - min_x) * cell_size
            sy = oy + (y - min_y) * cell_size
            for edge, neighbor in (
                ("left", (x - 1, y)),
                ("right", (x + 1, y)),
                ("top", (x, y - 1)),
                ("bottom", (x, y + 1)),
            ):
                if neighbor in piece:
                    continue
                if edge == "left":
                    parts.append(boundary_line(sx, sy, sx, sy + cell_size))
                elif edge == "right":
                    parts.append(boundary_line(sx + cell_size, sy, sx + cell_size, sy + cell_size))
                elif edge == "top":
                    parts.append(boundary_line(sx, sy, sx + cell_size, sy))
                else:
                    parts.append(boundary_line(sx, sy + cell_size, sx + cell_size, sy + cell_size))
        parts.append(svg_grid(min_x, min_y, max_x, max_y, cell_size, ox, oy))

    parts.append("</svg>\n")
    return "\n".join(parts)


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Generate half-cell polyomino-style tiling puzzle candidates."
    )
    parser.add_argument("--pieces", type=int, default=PIECE_COUNT)
    parser.add_argument("--board-w", type=int, default=BOARD_W)
    parser.add_argument("--board-h", type=int, default=BOARD_H)
    parser.add_argument("--min-solutions", type=int, default=TARGET_MIN_SOLUTIONS)
    parser.add_argument("--max-solutions", type=int, default=TARGET_MAX_SOLUTIONS)
    parser.add_argument("--allow-mirror", action="store_true", default=ALLOW_MIRROR)
    parser.add_argument("--no-rotate", action="store_true", default=not ALLOW_ROTATE)
    parser.add_argument("--seed", type=int, default=None)
    parser.add_argument("--candidates", type=int, default=20)
    parser.add_argument("--time-limit", type=float, default=300.0)
    parser.add_argument("--output-dir", type=Path, default=Path("out"))
    parser.add_argument("--allow-holes", action="store_true")
    parser.add_argument("--allow-identical-pieces", action="store_true")
    parser.add_argument("--min-half-cells", type=int, default=3)
    parser.add_argument("--solution-count-limit", type=int, default=SOLUTION_COUNT_LIMIT)
    parser.add_argument("--verbose", action="store_true")
    return parser.parse_args(argv)


def validate_args(args: argparse.Namespace) -> None:
    if args.pieces <= 0:
        raise SystemExit("--pieces must be positive")
    if args.pieces > len(LETTERS):
        raise SystemExit(f"--pieces must be at most {len(LETTERS)}")
    if args.board_w <= 0 or args.board_h <= 0:
        raise SystemExit("--board-w and --board-h must be positive")
    macro_area = args.pieces * (PIECE_AREA_SMALL // 4)
    if macro_area > args.board_w * args.board_h:
        raise SystemExit("board is too small for the requested piece count")
    if args.min_solutions > args.max_solutions:
        raise SystemExit("--min-solutions cannot exceed --max-solutions")
    if args.solution_count_limit < args.max_solutions:
        raise SystemExit("--solution-count-limit must be at least --max-solutions")
    if args.min_half_cells < 0:
        raise SystemExit("--min-half-cells must be non-negative")


def main(argv: list[str] | None = None) -> int:
    args = parse_args(argv)
    validate_args(args)
    candidates = generate_candidates(args)
    write_outputs(candidates, args.output_dir)
    if not candidates:
        print(
            "No candidate found. Try increasing --time-limit, changing --seed, "
            "or lowering --min-solutions.",
            file=sys.stderr,
        )
        return 1

    best = candidates[0]
    print(candidate_to_text(best, max_solutions=2))
    print(f"Saved {len(candidates)} candidate(s) to {args.output_dir}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile ideal_half_polyomino_search.py
#!/usr/bin/env python3
"""Heavy search for ideal half-cell puzzle candidates.

This script is intentionally stricter and heavier than generate_half_polyomino.py.
It searches for six non-identical pieces that tile a near-rectangular/full
rectangular board in at least K fixed-orientation ways.

Pipeline:
  1. Build many candidate piece shapes by generating legal tilings of a full
     rectangular board, then adding single-cell half transfers across region
     boundaries.  This produces pieces that are known to be practical tiling
     fragments rather than random disconnected-looking shapes.
  2. Enumerate every fixed-orientation placement of every candidate piece.
  3. Use OR-Tools CP-SAT to select six distinct pieces and K distinct exact
     covers of the same board.
  4. Verify the selected set with the existing exact-cover counter and write the
     normal JSON/SVG/HTML outputs.

Install:
  python -m pip install -r requirements.txt

Example:
  python ideal_half_polyomino_search.py --time-limit 1800 --library-target 2500 --output-dir out_ideal
"""

from __future__ import annotations

import argparse
import itertools
import json
import random
import sys
import time
from collections import defaultdict
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path

try:
    from ortools.sat.python import cp_model
except ModuleNotFoundError:  # pragma: no cover - user environment guard
    cp_model = None

import generate_half_polyomino as base


Cell = tuple[int, int]
MacroCell = tuple[int, int]


@dataclass(frozen=True)
class ShapeRecord:
    id: int
    cells: frozenset[Cell]
    area: int
    half_count: int
    fragile_count: int
    rotational_signature: tuple[Cell, ...]


@dataclass(frozen=True)
class PlacementRecord:
    id: int
    shape_id: int
    cells: frozenset[Cell]


@dataclass(frozen=True)
class CpsatSolutionProof:
    layer: int
    placements: tuple[PlacementRecord, ...]


@dataclass(frozen=True)
class CpsatCandidate:
    shapes: tuple[ShapeRecord, ...]
    proofs: tuple[CpsatSolutionProof, ...]


def align_cells(cells: set[Cell] | frozenset[Cell]) -> set[Cell]:
    """Normalize without shifting the 2x2 ordinary-cell grid out of phase."""
    min_x = min(x for x, _ in cells)
    min_y = min(y for _, y in cells)
    shift_x = min_x - (min_x % base.SCALE)
    shift_y = min_y - (min_y % base.SCALE)
    return {(x - shift_x, y - shift_y) for x, y in cells}


def exact_signature(cells: set[Cell] | frozenset[Cell]) -> tuple[Cell, ...]:
    return tuple(sorted(align_cells(cells)))


def rotational_signature(cells: set[Cell] | frozenset[Cell]) -> tuple[Cell, ...]:
    variants = []
    for rotation in range(4):
        oriented = align_cells(base.transform_cells(cells, rotation, False))
        if base.is_legal_half_cell_shape(oriented):
            variants.append(tuple(sorted(oriented)))
    if not variants:
        return tuple(sorted(align_cells(cells)))
    return min(variants)


def rectangle_board(width_small: int, height_small: int) -> set[Cell]:
    return {(x, y) for y in range(height_small) for x in range(width_small)}


def macro_rectangle(width_macro: int, height_macro: int) -> set[MacroCell]:
    return {(x, y) for y in range(height_macro) for x in range(width_macro)}


def macro_to_small_board(cells: set[MacroCell]) -> set[Cell]:
    return base.macro_to_full_small(cells)


def macro_perimeter(cells: set[MacroCell]) -> int:
    return sum(1 for cell in cells for neighbor in base.neighbors4(cell) if neighbor not in cells)


def board_score(cells: set[MacroCell], base_width: int, base_height: int) -> float:
    min_x = min(x for x, _ in cells)
    max_x = max(x for x, _ in cells)
    min_y = min(y for _, y in cells)
    max_y = max(y for _, y in cells)
    bbox_w = max_x - min_x + 1
    bbox_h = max_y - min_y + 1
    fill = len(cells) / (bbox_w * bbox_h)
    removed = base_width * base_height - len(cells)
    perimeter = macro_perimeter(cells)
    ideal_perimeter = 2 * (bbox_w + bbox_h)
    aspect = max(bbox_w, bbox_h) / max(1, min(bbox_w, bbox_h))
    return fill * 1000 - removed * 25 - (perimeter - ideal_perimeter) * 20 - max(0, aspect - 1.8) * 100


def boundary_macro_cells(cells: set[MacroCell]) -> list[MacroCell]:
    return sorted(
        cell
        for cell in cells
        if any(neighbor not in cells for neighbor in base.neighbors4(cell))
    )


def generate_near_rect_macro_boards(args: argparse.Namespace) -> list[set[MacroCell]]:
    base_board = macro_rectangle(args.board_w_macro, args.board_h_macro)
    candidates: list[set[MacroCell]] = []
    seen: set[tuple[MacroCell, ...]] = set()
    boundary = boundary_macro_cells(base_board)

    for remove_count in range(args.board_remove_min, args.board_remove_max + 1):
        if remove_count == 0:
            boards = [set(base_board)]
        else:
            combos = itertools.combinations(boundary, remove_count)
            boards = []
            for combo_index, combo in enumerate(combos):
                if combo_index >= args.max_board_candidates_per_remove:
                    break
                boards.append(base_board - set(combo))

        for board in boards:
            if len(board) == 0:
                continue
            signature = tuple(sorted(board))
            if signature in seen:
                continue
            seen.add(signature)
            if not base.is_macro_connected(board):
                continue
            if not args.allow_holes and base.has_macro_hole(board):
                continue
            small_area = len(board) * 4
            if not (args.pieces * args.min_piece_area <= small_area <= args.pieces * args.max_piece_area):
                continue
            candidates.append(board)

    candidates.sort(
        key=lambda board: board_score(board, args.board_w_macro, args.board_h_macro),
        reverse=True,
    )
    return candidates[: args.max_board_candidates]


def complement_half(mask: int) -> int:
    if mask == base.MASK_TOP:
        return base.MASK_BOTTOM
    if mask == base.MASK_BOTTOM:
        return base.MASK_TOP
    if mask == base.MASK_LEFT:
        return base.MASK_RIGHT
    if mask == base.MASK_RIGHT:
        return base.MASK_LEFT
    raise ValueError(f"not a half mask: {mask}")


def transfer_masks(direction: str, receiver_on_positive_side: bool) -> tuple[int, int]:
    """Return (donor_mask, receiver_mask) for one split ordinary cell."""
    if direction == "h":
        # Neighbor is horizontally adjacent.  Receiver gets the side touching it.
        receiver = base.MASK_RIGHT if receiver_on_positive_side else base.MASK_LEFT
    else:
        receiver = base.MASK_BOTTOM if receiver_on_positive_side else base.MASK_TOP
    return complement_half(receiver), receiver


def choose_area_pattern(
    rng: random.Random,
    piece_count: int,
    total_area: int,
    min_area: int,
    max_area: int,
) -> list[int]:
    allowed = [area for area in range(min_area, max_area + 1, 2)]
    for _ in range(10_000):
        areas = [rng.choice(allowed) for _ in range(piece_count - 1)]
        last = total_area - sum(areas)
        if last in allowed:
            areas.append(last)
            rng.shuffle(areas)
            return areas
    raise RuntimeError("could not create area pattern")


def choose_macro_region_sizes(
    rng: random.Random,
    board_area_macro: int,
    piece_count: int,
) -> list[int] | None:
    """Choose connected macro-region sizes before half-cell transfers.

    Region sizes of 3, 4, and 5 ordinary cells are useful because half transfers
    can turn them into 14..18 small-cell pieces.  This also supports boards such
    as 5x5 ordinary cells, whose area is not divisible by 4.
    """
    for _ in range(10_000):
        sizes = [rng.choice((3, 4, 5)) for _ in range(piece_count - 1)]
        last = board_area_macro - sum(sizes)
        if last in (3, 4, 5):
            sizes.append(last)
            rng.shuffle(sizes)
            return sizes
    return None


@lru_cache(maxsize=200_000)
def _connected_subsets_containing_cached(
    remaining_signature: tuple[MacroCell, ...],
    anchor: MacroCell,
    size: int,
    limit: int,
) -> tuple[tuple[MacroCell, ...], ...]:
    remaining = set(remaining_signature)
    subsets: set[frozenset[MacroCell]] = set()
    stack: list[tuple[frozenset[MacroCell], frozenset[MacroCell]]] = [
        (frozenset({anchor}), frozenset(n for n in base.neighbors4(anchor) if n in remaining))
    ]
    while stack and len(subsets) < limit:
        shape, frontier = stack.pop()
        if len(shape) == size:
            subsets.add(shape)
            continue
        for cell in list(frontier):
            new_shape = set(shape)
            new_shape.add(cell)
            new_frontier = set(frontier)
            new_frontier.remove(cell)
            for neighbor in base.neighbors4(cell):
                if neighbor in remaining and neighbor not in new_shape:
                    new_frontier.add(neighbor)
            stack.append((frozenset(new_shape), frozenset(new_frontier)))
    return tuple(tuple(sorted(subset)) for subset in subsets)


def connected_subsets_containing(
    remaining: set[MacroCell],
    anchor: MacroCell,
    size: int,
    limit: int = 400,
) -> list[frozenset[MacroCell]]:
    return [
        frozenset(subset)
        for subset in _connected_subsets_containing_cached(
            tuple(sorted(remaining)), anchor, size, limit
        )
    ]


def partition_macro_board(
    rng: random.Random,
    board: set[MacroCell],
    piece_count: int,
    max_nodes: int = 20_000,
) -> list[set[MacroCell]] | None:
    sizes = choose_macro_region_sizes(rng, len(board), piece_count)
    if sizes is None:
        return None
    sizes.sort(reverse=True)
    nodes = 0

    def search(
        remaining: frozenset[MacroCell],
        remaining_sizes: tuple[int, ...],
    ) -> list[set[MacroCell]] | None:
        nonlocal nodes
        nodes += 1
        if nodes > max_nodes:
            return None
        if not remaining_sizes:
            return [] if not remaining else None
        if sum(remaining_sizes) != len(remaining):
            return None

        size = remaining_sizes[0]
        anchor = min(remaining)
        options = connected_subsets_containing(set(remaining), anchor, size)
        rng.shuffle(options)
        for subset in options:
            rest = frozenset(set(remaining) - set(subset))
            if rest and not base.is_macro_connected(set(rest)):
                # A disconnected remainder can never be fully covered by
                # connected regions without crossing occupied cells.
                continue
            result = search(rest, remaining_sizes[1:])
            if result is not None:
                return [set(subset)] + result
        return None

    return search(frozenset(board), tuple(sizes))


def random_tetromino_partition(
    rng: random.Random,
    board: set[MacroCell],
    piece_count: int,
) -> list[set[MacroCell]] | None:
    if len(board) == piece_count * 4:
        return base.partition_into_tetrominoes(board, piece_count, rng, max_nodes=100_000)
    return partition_macro_board(rng, board, piece_count)


def boundary_edges(regions: list[set[MacroCell]]) -> list[tuple[MacroCell, MacroCell, int, int, str]]:
    owner: dict[MacroCell, int] = {}
    for index, region in enumerate(regions):
        for cell in region:
            owner[cell] = index

    edges: list[tuple[MacroCell, MacroCell, int, int, str]] = []
    for cell, p in owner.items():
        x, y = cell
        for neighbor, direction in (((x + 1, y), "h"), ((x, y + 1), "v")):
            q = owner.get(neighbor)
            if q is not None and q != p:
                edges.append((cell, neighbor, p, q, direction))
    return edges


def make_partition_shapes(
    regions: list[set[MacroCell]],
    rng: random.Random,
    min_area: int,
    max_area: int,
    min_half_cells: int,
    max_fragile: int,
    transfer_attempts: int,
) -> list[set[Cell]] | None:
    edges = boundary_edges(regions)
    if not edges:
        return None
    valid_pieces: list[set[Cell]] = []
    seen: set[tuple[Cell, ...]] = set()
    for _ in range(transfer_attempts):
        masks_by_piece = [{cell: base.MASK_FULL for cell in region} for region in regions]
        areas = [len(region) * 4 for region in regions]
        split_cells: set[MacroCell] = set()
        shuffled_edges = edges[:]
        rng.shuffle(shuffled_edges)

        # Split many boundary ordinary-cells.  Each split transfers one half-cell
        # from the original owner to the neighboring piece.  Areas may end at
        # 14, 16, or 18 small cells by default.
        split_budget = rng.randint(
            max(3, min_half_cells * len(regions) // 2),
            max(3, len(edges)),
        )
        for c, d, p, q, direction in shuffled_edges:
            if len(split_cells) >= split_budget:
                break
            options = [(c, p, q, True), (d, q, p, False)]
            rng.shuffle(options)
            for cell, donor, receiver, positive in options:
                if cell in split_cells:
                    continue
                if areas[donor] - 2 < min_area:
                    continue
                if areas[receiver] + 2 > max_area:
                    continue
                donor_mask, receiver_mask = transfer_masks(direction, positive)
                masks_by_piece[donor][cell] = donor_mask
                masks_by_piece[receiver][cell] = receiver_mask
                areas[donor] -= 2
                areas[receiver] += 2
                split_cells.add(cell)
                break

        pieces = [align_cells(base.masks_to_cells(masks)) for masks in masks_by_piece]
        for piece in pieces:
            if not validate_ideal_piece(piece, min_area, max_area, min_half_cells, max_fragile):
                continue
            signature = exact_signature(piece)
            if signature in seen:
                continue
            seen.add(signature)
            valid_pieces.append(piece)
            if len(valid_pieces) >= len(regions):
                return valid_pieces
    return valid_pieces or None


def validate_ideal_piece(
    cells: set[Cell],
    min_area: int,
    max_area: int,
    min_half_cells: int,
    max_fragile: int,
) -> bool:
    if not (min_area <= len(cells) <= max_area):
        return False
    if len(cells) % 2 != 0:
        return False
    if not base.is_connected(cells):
        return False
    if not base.is_legal_half_cell_shape(cells):
        return False
    if base.count_half_cells(cells) < min_half_cells:
        return False
    if base.count_quarter_artifacts(cells) != 0:
        return False
    if base.count_fragile_artifacts(cells) > max_fragile:
        return False
    return True


def build_shape_library(args: argparse.Namespace, macro_board: set[MacroCell]) -> list[ShapeRecord]:
    rng = random.Random(args.seed)
    records: list[ShapeRecord] = []
    exact_seen: set[tuple[Cell, ...]] = set()
    rotational_seen_count: defaultdict[tuple[Cell, ...], int] = defaultdict(int)
    start = time.monotonic()
    attempts = 0

    while len(records) < args.library_target:
        attempts += 1
        if time.monotonic() - start > args.library_time_limit:
            break

        regions = random_tetromino_partition(
            rng, macro_board, args.pieces
        )
        if regions is None:
            continue

        pieces = make_partition_shapes(
            regions,
            rng,
            min_area=args.min_piece_area,
            max_area=args.max_piece_area,
            min_half_cells=args.min_half_cells,
            max_fragile=args.max_fragile,
            transfer_attempts=args.transfer_attempts,
        )
        if pieces is None:
            continue

        for piece in pieces:
            signature = exact_signature(piece)
            if signature in exact_seen:
                continue
            rot_sig = rotational_signature(piece)
            if rotational_seen_count[rot_sig] >= args.max_per_rotational_family:
                continue
            exact_seen.add(signature)
            rotational_seen_count[rot_sig] += 1
            cells = frozenset(piece)
            records.append(
                ShapeRecord(
                    id=len(records),
                    cells=cells,
                    area=len(cells),
                    half_count=base.count_half_cells(cells),
                    fragile_count=base.count_fragile_artifacts(cells),
                    rotational_signature=rot_sig,
                )
            )
            if len(records) >= args.library_target:
                break

        if args.verbose and attempts % 100 == 0:
            print(
                f"library attempts={attempts} shapes={len(records)} "
                f"elapsed={time.monotonic() - start:.1f}s",
                file=sys.stderr,
                flush=True,
            )

    if getattr(args, "allow_identical_pieces", False) and getattr(args, "shape_copies", 1) > 1:
        originals = records[:]
        for _ in range(1, args.shape_copies):
            for record in originals:
                records.append(
                    ShapeRecord(
                        id=len(records),
                        cells=record.cells,
                        area=record.area,
                        half_count=record.half_count,
                        fragile_count=record.fragile_count,
                        rotational_signature=record.rotational_signature,
                    )
                )
    return records


def enumerate_shape_placements(
    board: set[Cell],
    shapes: list[ShapeRecord],
) -> list[PlacementRecord]:
    placements: list[PlacementRecord] = []
    board_set = set(board)
    _, _, board_max_x, board_max_y = base.bounds(board_set)
    seen: set[tuple[int, tuple[Cell, ...]]] = set()

    for shape in shapes:
        _, _, shape_max_x, shape_max_y = base.bounds(shape.cells)
        for ty in range(0, board_max_y - shape_max_y + 1, base.SCALE):
            for tx in range(0, board_max_x - shape_max_x + 1, base.SCALE):
                placed = frozenset((x + tx, y + ty) for x, y in shape.cells)
                if placed <= board_set:
                    key = (shape.id, tuple(sorted(placed)))
                    if key in seen:
                        continue
                    seen.add(key)
                    placements.append(
                        PlacementRecord(
                            id=len(placements),
                            shape_id=shape.id,
                            cells=placed,
                        )
                    )
    return placements


def enumerate_phase_preserving_placements(
    board: set[Cell],
    pieces: list[set[Cell]],
) -> tuple[list[list[base.Placement]], dict[Cell, int], int]:
    board_cells = sorted(board)
    index = {cell: i for i, cell in enumerate(board_cells)}
    board_mask = (1 << len(board_cells)) - 1
    _, _, board_max_x, board_max_y = base.bounds(board)

    placements_by_piece: list[list[base.Placement]] = []
    for piece_index, piece in enumerate(pieces):
        aligned = align_cells(piece)
        _, _, piece_max_x, piece_max_y = base.bounds(aligned)
        piece_placements: list[base.Placement] = []
        for ty in range(0, board_max_y - piece_max_y + 1, base.SCALE):
            for tx in range(0, board_max_x - piece_max_x + 1, base.SCALE):
                placed = frozenset((x + tx, y + ty) for x, y in aligned)
                if not placed <= board:
                    continue
                mask = 0
                for cell in placed:
                    mask |= 1 << index[cell]
                piece_placements.append(
                    base.Placement(
                        piece_index=piece_index,
                        cells=placed,
                        mask=mask,
                        origin=(tx, ty),
                        orientation=0,
                    )
                )
        unique: dict[int, base.Placement] = {}
        for placement in piece_placements:
            unique.setdefault(placement.mask, placement)
        placements_by_piece.append(list(unique.values()))
    return placements_by_piece, index, board_mask


def count_solutions_fixed_phase(
    board: set[Cell],
    pieces: list[set[Cell]],
    limit: int,
) -> tuple[int, list[dict[int, frozenset[Cell]]], list[list[base.Placement]]]:
    placements_by_piece, _index, board_mask = enumerate_phase_preserving_placements(board, pieces)
    if any(not placements for placements in placements_by_piece):
        return 0, [], placements_by_piece

    cell_to_placements: dict[int, list[tuple[int, base.Placement]]] = defaultdict(list)
    for piece_index, placements_for_piece in enumerate(placements_by_piece):
        for placement in placements_for_piece:
            m = placement.mask
            while m:
                bit = m & -m
                cell_index = bit.bit_length() - 1
                cell_to_placements[cell_index].append((piece_index, placement))
                m ^= bit

    remaining_start = frozenset(range(len(pieces)))
    solutions: list[dict[int, frozenset[Cell]]] = []
    count = 0

    def search(occupied: int, remaining: frozenset[int], chosen: dict[int, base.Placement]) -> None:
        nonlocal count
        if count >= limit:
            return
        if not remaining:
            if occupied == board_mask:
                count += 1
                if len(solutions) < limit:
                    solutions.append({p: placement.cells for p, placement in chosen.items()})
            return

        empty_mask = board_mask & ~occupied
        best_cell_index = None
        best_options: list[tuple[int, base.Placement]] | None = None
        m = empty_mask
        while m:
            bit = m & -m
            cell_index = bit.bit_length() - 1
            options = [
                (p, placement)
                for p, placement in cell_to_placements[cell_index]
                if p in remaining and (placement.mask & occupied) == 0
            ]
            if not options:
                return
            if best_options is None or len(options) < len(best_options):
                best_cell_index = cell_index
                best_options = options
                if len(options) == 1:
                    break
            m ^= bit

        assert best_cell_index is not None and best_options is not None
        best_options.sort(key=lambda item: (item[0], item[1].origin, item[1].orientation))
        for piece_index, placement in best_options:
            chosen[piece_index] = placement
            search(
                occupied | placement.mask,
                frozenset(p for p in remaining if p != piece_index),
                chosen,
            )
            chosen.pop(piece_index, None)
            if count >= limit:
                return

    search(0, remaining_start, {})
    return count, solutions, placements_by_piece


def _build_cpsat_model(
    board: set[Cell],
    shapes: list[ShapeRecord],
    placements: list[PlacementRecord],
    args: argparse.Namespace,
) -> tuple[
    cp_model.CpModel,
    dict[int, cp_model.IntVar],
    dict[tuple[int, int], cp_model.IntVar],
    dict[int, PlacementRecord],
] | None:
    if cp_model is None:
        raise SystemExit(
            "OR-Tools is not installed. Run: python -m pip install -r requirements.txt"
        )
    if not placements:
        return None

    model = cp_model.CpModel()
    k_solutions = args.required_solutions
    y = {shape.id: model.NewBoolVar(f"shape_{shape.id}") for shape in shapes}
    x: dict[tuple[int, int], cp_model.IntVar] = {}
    for layer in range(k_solutions):
        for placement in placements:
            x[(layer, placement.id)] = model.NewBoolVar(f"x_{layer}_{placement.id}")

    placements_by_shape: defaultdict[int, list[PlacementRecord]] = defaultdict(list)
    placements_by_cell: defaultdict[Cell, list[PlacementRecord]] = defaultdict(list)
    for placement in placements:
        placements_by_shape[placement.shape_id].append(placement)
        for cell in placement.cells:
            placements_by_cell[cell].append(placement)

    model.Add(sum(y.values()) == args.pieces)
    model.Add(sum(shape.area * y[shape.id] for shape in shapes) == len(board))

    # Usually we avoid selecting two shapes that are the same physical cut up to
    # rotation.  Ranked search may allow them, but downstream verification counts
    # solutions modulo pure identical-piece swaps.
    if not getattr(args, "allow_identical_pieces", False):
        by_rotational: defaultdict[tuple[Cell, ...], list[int]] = defaultdict(list)
        for shape in shapes:
            by_rotational[shape.rotational_signature].append(shape.id)
        for ids in by_rotational.values():
            if len(ids) > 1:
                model.Add(sum(y[i] for i in ids) <= 1)

    for layer in range(k_solutions):
        for cell in board:
            covering = [x[(layer, p.id)] for p in placements_by_cell[cell]]
            if not covering:
                return None
            model.Add(sum(covering) == 1)

        for shape in shapes:
            model.Add(
                sum(x[(layer, p.id)] for p in placements_by_shape[shape.id])
                == y[shape.id]
            )

    # Make every requested solution genuinely different from every other one.
    for a in range(k_solutions):
        for b in range(a + 1, k_solutions):
            same_vars = []
            for placement in placements:
                z = model.NewBoolVar(f"same_{a}_{b}_{placement.id}")
                model.Add(z <= x[(a, placement.id)])
                model.Add(z <= x[(b, placement.id)])
                model.Add(z >= x[(a, placement.id)] + x[(b, placement.id)] - 1)
                same_vars.append(z)
            model.Add(sum(same_vars) <= args.pieces - 1)

    # Prefer more half-cell structure and cleaner pieces.
    model.Maximize(
        sum(shape.half_count * y[shape.id] for shape in shapes)
        - sum(shape.fragile_count * 20 * y[shape.id] for shape in shapes)
    )
    return model, y, x, {placement.id: placement for placement in placements}


def solve_with_cpsat_candidates(
    board: set[Cell],
    shapes: list[ShapeRecord],
    placements: list[PlacementRecord],
    args: argparse.Namespace,
    max_candidates: int = 1,
) -> list[CpsatCandidate]:
    built = _build_cpsat_model(board, shapes, placements, args)
    if built is None:
        return []

    model, y, x, placement_by_id = built
    selected_by_id = {shape.id: shape for shape in shapes}
    start = time.monotonic()
    results: list[CpsatCandidate] = []
    seen: set[tuple[int, ...]] = set()

    for attempt in range(max(1, max_candidates)):
        remaining_time = args.solve_time_limit - (time.monotonic() - start)
        if remaining_time <= 0:
            break

        solver = cp_model.CpSolver()
        solver.parameters.max_time_in_seconds = remaining_time
        solver.parameters.num_search_workers = args.workers
        solver.parameters.random_seed = (args.seed or 1) + attempt
        solver.parameters.log_search_progress = bool(getattr(args, "solver_log", False) and attempt == 0)

        status = solver.Solve(model)
        if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
            break

        selected_ids = tuple(sorted(shape_id for shape_id in y if solver.Value(y[shape_id]) == 1))
        if selected_ids in seen:
            break
        seen.add(selected_ids)
        selected_shapes = tuple(selected_by_id[i] for i in selected_ids)

        proofs: list[CpsatSolutionProof] = []
        for layer in range(args.required_solutions):
            chosen = tuple(
                placement_by_id[placement_id]
                for placement_id in sorted(placement_by_id)
                if solver.Value(x[(layer, placement_id)]) == 1
            )
            proofs.append(CpsatSolutionProof(layer=layer, placements=chosen))
        results.append(CpsatCandidate(shapes=selected_shapes, proofs=tuple(proofs)))

        # Ask CP-SAT for a genuinely different set of physical cuts next time.
        model.Add(sum(y[i] for i in selected_ids) <= args.pieces - 1)

    return results


def solve_with_cpsat(
    board: set[Cell],
    shapes: list[ShapeRecord],
    placements: list[PlacementRecord],
    args: argparse.Namespace,
) -> list[ShapeRecord] | None:
    candidates = solve_with_cpsat_candidates(board, shapes, placements, args, max_candidates=1)
    if not candidates:
        return None

    return list(candidates[0].shapes)


def verify_cpsat_proof_cover(
    board: set[Cell],
    candidate: CpsatCandidate,
    args: argparse.Namespace,
) -> tuple[bool, str]:
    if not candidate.proofs:
        return False, "no proof layers"
    selected_shape_ids = {shape.id for shape in candidate.shapes}
    board_set = set(board)

    for proof in candidate.proofs:
        if len(proof.placements) != args.pieces:
            return (
                False,
                f"proof layer {proof.layer} failed: used {len(proof.placements)} placements, expected {args.pieces}",
            )
        used_shape_ids = {placement.shape_id for placement in proof.placements}
        if used_shape_ids != selected_shape_ids:
            return (
                False,
                f"proof layer {proof.layer} failed: used shape ids differ from selected shape ids",
            )

        covered: set[Cell] = set()
        for placement in proof.placements:
            if not placement.cells <= board_set:
                outside = sorted(placement.cells - board_set)[:1]
                return (
                    False,
                    f"proof layer {proof.layer} failed: placement {placement.id} outside board at {outside}",
                )
            overlap = covered & placement.cells
            if overlap:
                return (
                    False,
                    f"proof layer {proof.layer} failed: overlap at cell {sorted(overlap)[0]}",
                )
            covered |= set(placement.cells)
        if covered != board_set:
            return (
                False,
                f"proof layer {proof.layer} failed: covered {len(covered)} cells but board has {len(board_set)}",
            )
    return True, f"{len(candidate.proofs)} proof layer(s) cover the board"


def verify_and_write(
    board: set[Cell],
    selected_shapes: list[ShapeRecord],
    args: argparse.Namespace,
) -> bool:
    pieces = [set(shape.cells) for shape in selected_shapes]
    solution_count, solutions, placements_by_piece = base.count_solutions(
        board,
        pieces,
        allow_rotate=False,
        allow_mirror=False,
        limit=args.solution_count_limit,
    )
    if solution_count < args.required_solutions:
        print(
            f"CP-SAT candidate failed verification: only {solution_count} solutions",
            file=sys.stderr,
        )
        return False

    rotated_count, _, _ = base.count_solutions(
        board,
        pieces,
        allow_rotate=not args.no_rotate,
        allow_mirror=False,
        limit=args.solution_count_limit,
    )
    analysis = base.analyze_candidate(
        board,
        pieces,
        solutions,
        solution_count,
        rotated_count,
        placements_by_piece,
        args.required_solutions,
        args.max_solutions,
    )
    candidate = base.Candidate(
        board=board,
        pieces=pieces,
        solutions=solutions,
        solution_count=solution_count,
        placements_by_piece=placements_by_piece,
        score=analysis.difficulty_score,
        analysis=analysis,
        attempts=0,
    )
    base.write_outputs([candidate], args.output_dir)
    print(base.candidate_to_text(candidate, max_solutions=min(4, len(solutions))))
    print(f"Saved ideal candidate to {args.output_dir}")
    return True


def write_library_snapshot(
    shapes: list[ShapeRecord],
    placements: list[PlacementRecord],
    path: Path,
) -> None:
    payload = {
        "shape_count": len(shapes),
        "placement_count": len(placements),
        "shapes": [
            {
                "id": shape.id,
                "area": shape.area,
                "half_count": shape.half_count,
                "fragile_count": shape.fragile_count,
                "cells": sorted([list(cell) for cell in shape.cells]),
            }
            for shape in shapes
        ],
    }
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Heavy ideal half-polyomino search.")
    parser.add_argument("--pieces", type=int, default=6)
    parser.add_argument("--board-w-macro", type=int, default=6)
    parser.add_argument("--board-h-macro", type=int, default=4)
    parser.add_argument("--board-remove-min", type=int, default=0)
    parser.add_argument("--board-remove-max", type=int, default=0)
    parser.add_argument("--max-board-candidates", type=int, default=20)
    parser.add_argument("--max-board-candidates-per-remove", type=int, default=5000)
    parser.add_argument("--allow-holes", action="store_true")
    parser.add_argument("--min-piece-area", type=int, default=14)
    parser.add_argument("--max-piece-area", type=int, default=18)
    parser.add_argument("--min-half-cells", type=int, default=3)
    parser.add_argument("--max-fragile", type=int, default=0)
    parser.add_argument("--required-solutions", type=int, default=4)
    parser.add_argument("--max-solutions", type=int, default=16)
    parser.add_argument("--solution-count-limit", type=int, default=100)
    parser.add_argument("--library-target", type=int, default=2500)
    parser.add_argument("--library-time-limit", type=float, default=900.0)
    parser.add_argument("--solve-time-limit", type=float, default=1800.0)
    parser.add_argument("--transfer-attempts", type=int, default=200)
    parser.add_argument("--max-per-rotational-family", type=int, default=1)
    parser.add_argument("--allow-identical-pieces", action="store_true")
    parser.add_argument("--shape-copies", type=int, default=2)
    parser.add_argument("--workers", type=int, default=8)
    parser.add_argument("--seed", type=int, default=1)
    parser.add_argument("--output-dir", type=Path, default=Path("out_ideal"))
    parser.add_argument("--library-json", type=Path, default=Path("out_ideal/library.json"))
    parser.add_argument("--no-rotate", action="store_true")
    parser.add_argument("--verbose", action="store_true")
    parser.add_argument("--solver-log", action="store_true")
    return parser.parse_args(argv)


def validate_args(args: argparse.Namespace) -> None:
    if args.pieces <= 0:
        raise SystemExit("--pieces must be positive")
    if args.board_remove_min < 0 or args.board_remove_max < 0:
        raise SystemExit("board remove counts must be non-negative")
    if args.board_remove_min > args.board_remove_max:
        raise SystemExit("--board-remove-min cannot exceed --board-remove-max")
    min_board_area = (args.board_w_macro * args.board_h_macro - args.board_remove_max) * 4
    max_board_area = (args.board_w_macro * args.board_h_macro - args.board_remove_min) * 4
    if max_board_area < args.pieces * args.min_piece_area:
        raise SystemExit("board area is too small for piece area bounds")
    if min_board_area > args.pieces * args.max_piece_area:
        raise SystemExit("board area is too large for piece area bounds")
    if args.min_piece_area % 2 or args.max_piece_area % 2:
        raise SystemExit("piece area bounds must be even small-cell counts")
    if args.required_solutions < 2:
        raise SystemExit("--required-solutions should be at least 2")
    if args.solution_count_limit < args.required_solutions:
        raise SystemExit("--solution-count-limit must be at least --required-solutions")


def main(argv: list[str] | None = None) -> int:
    args = parse_args(argv)
    validate_args(args)
    if cp_model is None:
        print("OR-Tools is not installed.", file=sys.stderr)
        print("Run: python -m pip install -r requirements.txt", file=sys.stderr)
        return 2

    args.output_dir.mkdir(parents=True, exist_ok=True)
    args.library_json.parent.mkdir(parents=True, exist_ok=True)

    macro_boards = generate_near_rect_macro_boards(args)
    print(f"Generated {len(macro_boards)} near-rect board(s).", file=sys.stderr, flush=True)
    if not macro_boards:
        print("No board candidates match the area and shape constraints.", file=sys.stderr)
        return 1

    for board_index, macro_board in enumerate(macro_boards, start=1):
        board = macro_to_small_board(macro_board)
        board_dir = args.output_dir / f"board_{board_index:03d}"
        library_json = board_dir / "library.json"
        board_dir.mkdir(parents=True, exist_ok=True)
        print(
            f"[board {board_index}/{len(macro_boards)}] "
            f"macro_area={len(macro_board)} small_area={len(board)} "
            f"score={board_score(macro_board, args.board_w_macro, args.board_h_macro):.1f}",
            file=sys.stderr,
            flush=True,
        )

        print("Building shape library...", file=sys.stderr, flush=True)
        shapes = build_shape_library(args, macro_board)
        print(f"Built {len(shapes)} shapes.", file=sys.stderr, flush=True)
        if len(shapes) < args.pieces:
            print("Not enough shapes for this board.", file=sys.stderr)
            continue

        print("Enumerating placements...", file=sys.stderr, flush=True)
        placements = enumerate_shape_placements(board, shapes)
        print(f"Enumerated {len(placements)} placements.", file=sys.stderr, flush=True)
        write_library_snapshot(shapes, placements, library_json)

        print("Solving CP-SAT exact-cover selection...", file=sys.stderr, flush=True)
        selected = solve_with_cpsat(board, shapes, placements, args)
        if selected is None:
            print("No ideal candidate found for this board.", file=sys.stderr)
            continue

        old_output_dir = args.output_dir
        args.output_dir = board_dir
        ok = verify_and_write(board, selected, args)
        args.output_dir = old_output_dir
        if ok:
            print(f"Found candidate in {board_dir}", file=sys.stderr)
            return 0

    print("No ideal candidate found under the current constraints.", file=sys.stderr)
    return 1


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile ranked_ideal_search.py
#!/usr/bin/env python3
"""One-shot ranked search for robust half-cell puzzle candidates.

This runner keeps the non-negotiable constraints hard, then ranks the rest.

Hard filters:
  - legal half-cell masks only; no quarter/three-quarter/diagonal/L masks
  - paper fragility count must be 0
  - effective fixed-orientation solution count must be high enough

Important scoring rule:
  Identical physical pieces are allowed, but pure swaps of identical pieces are
  collapsed before counting solutions. Duplicate pieces are a penalty, not a
  hard rejection unless --no-identical-pieces is used.
"""

from __future__ import annotations

import argparse
import copy
import json
import sys
import time
from collections import defaultdict
from dataclasses import dataclass
from html import escape
from pathlib import Path

import generate_half_polyomino as base
import ideal_half_polyomino_search as ideal


@dataclass
class RankedResult:
    candidate: base.Candidate
    board_index: int
    profile_name: str
    rank_score: float
    notes: list[str]


@dataclass
class NearMiss:
    candidate: base.Candidate
    reject_reason: str
    raw_solution_count: int
    effective_solution_count: int
    proof_layer_count: int
    proof_valid: bool
    proof_message: str
    profile_name: str
    board_index: int
    seed: int
    selected_shape_ids: list[int]
    selected_shape_cells: list[list[tuple[int, int]]]
    board_metrics: dict[str, float]
    placements_by_piece_counts: list[int]


def macro_bounds(cells: set[tuple[int, int]]) -> tuple[int, int, int, int]:
    min_x = min(x for x, _ in cells)
    max_x = max(x for x, _ in cells)
    min_y = min(y for _, y in cells)
    max_y = max(y for _, y in cells)
    return min_x, min_y, max_x, max_y


def small_board_metrics(board: set[tuple[int, int]]) -> dict[str, float]:
    macro = {(x // base.SCALE, y // base.SCALE) for x, y in board}
    min_x, min_y, max_x, max_y = macro_bounds(macro)
    bbox_w = max_x - min_x + 1
    bbox_h = max_y - min_y + 1
    fill = len(macro) / (bbox_w * bbox_h)
    perimeter = ideal.macro_perimeter(macro)
    ideal_perimeter = 2 * (bbox_w + bbox_h)
    aspect = max(bbox_w, bbox_h) / max(1, min(bbox_w, bbox_h))
    return {
        "macro_area": len(macro),
        "bbox_w": bbox_w,
        "bbox_h": bbox_h,
        "fill": fill,
        "perimeter_extra": max(0, perimeter - ideal_perimeter),
        "aspect": aspect,
    }


def unique_effective_solutions(
    solutions: list[dict[int, frozenset[tuple[int, int]]]],
    pieces: list[set[tuple[int, int]]],
    limit: int,
) -> list[dict[int, frozenset[tuple[int, int]]]]:
    seen: set[object] = set()
    unique: list[dict[int, frozenset[tuple[int, int]]]] = []
    for solution in solutions:
        signature = base.solution_identity_signature(solution, pieces)
        if signature in seen:
            continue
        seen.add(signature)
        unique.append(solution)
        if len(unique) >= limit:
            break
    return unique


def rank_candidate(candidate: base.Candidate, profile_name: str) -> tuple[float, list[str]]:
    analysis = candidate.analysis
    metrics = small_board_metrics(candidate.board)
    pieces = candidate.pieces
    areas = [len(piece) for piece in pieces]
    half_counts = [base.count_half_cells(piece) for piece in pieces]

    score = 0.0
    notes: list[str] = []

    if analysis.fragile_artifact_count != 0:
        score -= 100_000
        notes.append("fragile>0")
    if analysis.quarter_artifact_count != 0:
        score -= 100_000
        notes.append("quarter>0")

    if analysis.solution_count >= 4:
        score += 280
        notes.append(f"effective {analysis.solution_count} fixed solutions")
    elif analysis.solution_count == 3:
        score += 150
        notes.append("effective 3 fixed solutions")
    elif analysis.solution_count == 2:
        score += 75
        notes.append("effective 2 fixed solutions")
    else:
        score -= 500
        notes.append("too few fixed solutions")

    score += metrics["fill"] * 540
    score -= metrics["perimeter_extra"] * 38
    score -= max(0.0, metrics["aspect"] - 1.55) * 190
    score -= abs(metrics["macro_area"] - 24) * 8

    total_half = sum(half_counts)
    score += min(total_half, 26) * 12
    score += min(half_counts) * 22

    score -= (max(areas) - min(areas)) * 8
    score -= sum(abs(area - 16) for area in areas) * 2

    if analysis.duplicate_piece_count:
        score -= analysis.duplicate_piece_count * 55
        notes.append(f"duplicate pieces {analysis.duplicate_piece_count}")

    avg_candidates = analysis.average_piece_candidates
    if avg_candidates < 2:
        score -= 100
        notes.append("few placement choices")
    elif avg_candidates > 45:
        score -= 35
        notes.append("many placement choices")
    else:
        score += min(avg_candidates, 18) * 4

    notes.append(
        f"board {int(metrics['bbox_w'])}x{int(metrics['bbox_h'])}, "
        f"fill={metrics['fill']:.2f}, extra_perimeter={int(metrics['perimeter_extra'])}"
    )
    notes.append(f"half cells total {total_half}")
    notes.append(profile_name)
    return score, notes


def make_candidate(
    board: set[tuple[int, int]],
    selected: ideal.CpsatCandidate,
    args: argparse.Namespace,
) -> tuple[base.Candidate | None, NearMiss | None, str]:
    pieces = [set(shape.cells) for shape in selected.shapes]
    proof_valid, proof_message = ideal.verify_cpsat_proof_cover(board, selected, args)
    raw_solution_count, raw_solutions, placements_by_piece = ideal.count_solutions_fixed_phase(
        board,
        pieces,
        limit=args.solution_count_limit,
    )

    effective_solutions = unique_effective_solutions(
        raw_solutions,
        pieces,
        limit=args.solution_count_limit,
    )
    effective_solution_count = len(effective_solutions)

    rotated_count, _, _ = base.count_solutions(
        board,
        pieces,
        allow_rotate=not args.no_rotate,
        allow_mirror=False,
        limit=args.solution_count_limit,
    )
    analysis = base.analyze_candidate(
        board,
        pieces,
        effective_solutions,
        effective_solution_count,
        rotated_count,
        placements_by_piece,
        args.min_acceptable_solutions,
        args.target_solutions,
    )

    candidate = base.Candidate(
        board=board,
        pieces=pieces,
        solutions=effective_solutions,
        solution_count=effective_solution_count,
        placements_by_piece=placements_by_piece,
        score=analysis.difficulty_score,
        analysis=analysis,
        attempts=0,
    )

    reject_reasons: list[str] = []
    if not proof_valid:
        reject_reasons.append(f"proof_invalid:{proof_message}")
    if raw_solution_count == 0:
        reject_reasons.append("raw_solution_count=0")
    if effective_solution_count < args.min_acceptable_solutions:
        reject_reasons.append(
            f"effective_solution_count={effective_solution_count}<min={args.min_acceptable_solutions}"
        )
    if analysis.quarter_artifact_count != 0:
        reject_reasons.append(f"quarter_artifact_count={analysis.quarter_artifact_count}")
    if analysis.fragile_artifact_count != 0:
        reject_reasons.append(f"fragile_artifact_count={analysis.fragile_artifact_count}")
    if analysis.duplicate_piece_count != 0 and not args.allow_identical_pieces:
        reject_reasons.append(f"duplicate_piece_count={analysis.duplicate_piece_count}")

    near_miss = NearMiss(
        candidate=candidate,
        reject_reason="; ".join(reject_reasons) if reject_reasons else "",
        raw_solution_count=raw_solution_count,
        effective_solution_count=effective_solution_count,
        proof_layer_count=len(selected.proofs),
        proof_valid=proof_valid,
        proof_message=proof_message,
        profile_name=getattr(args, "profile_name", ""),
        board_index=getattr(args, "board_index", 0),
        seed=args.seed,
        selected_shape_ids=[shape.id for shape in selected.shapes],
        selected_shape_cells=[sorted(shape.cells) for shape in selected.shapes],
        board_metrics=small_board_metrics(board),
        placements_by_piece_counts=[len(placements) for placements in placements_by_piece],
    )

    if reject_reasons:
        return None, near_miss, near_miss.reject_reason

    return candidate, None, "accepted"


def should_store_near_miss(miss: NearMiss, args: argparse.Namespace) -> bool:
    if not args.write_near_misses:
        return False
    min_solutions = getattr(args, "min_acceptable_solutions", getattr(args, "min_solutions", 2))
    if miss.proof_valid and miss.effective_solution_count < min_solutions:
        return True
    if miss.raw_solution_count >= 1 and miss.effective_solution_count < min_solutions:
        return True
    if args.accept_one_solution_nearmiss and miss.effective_solution_count >= 1:
        return True
    if miss.candidate.analysis.fragile_artifact_count == 1:
        return True
    if "duplicate_piece_count" in miss.reject_reason:
        return True
    if miss.board_metrics["fill"] >= 0.88 and miss.effective_solution_count < min_solutions:
        return True
    return False


def add_near_miss(near_misses: list[NearMiss], miss: NearMiss | None, args: argparse.Namespace) -> None:
    if miss is None or not should_store_near_miss(miss, args):
        return
    near_misses.append(miss)
    near_misses.sort(
        key=lambda item: (
            item.effective_solution_count,
            item.raw_solution_count,
            item.board_metrics["fill"],
            -item.board_metrics["perimeter_extra"],
        ),
        reverse=True,
    )
    del near_misses[args.near_miss_limit :]


def trim_solver_input(
    shapes: list[ideal.ShapeRecord],
    placements: list[ideal.PlacementRecord],
    args: argparse.Namespace,
) -> tuple[list[ideal.ShapeRecord], list[ideal.PlacementRecord]]:
    if not shapes or not placements:
        return shapes, placements
    if len(shapes) <= args.max_solver_shapes and len(placements) <= args.max_solver_placements:
        return shapes, placements

    placements_by_shape: defaultdict[int, int] = defaultdict(int)
    for placement in placements:
        placements_by_shape[placement.shape_id] += 1

    def shape_key(shape: ideal.ShapeRecord) -> tuple[float, int, int, int]:
        placement_count = placements_by_shape[shape.id]
        if placement_count == 0:
            return (-1_000_000, 0, 0, 0)
        flexible_bonus = min(placement_count, 28)
        overflex_penalty = max(0, placement_count - 45)
        return (
            shape.half_count * 100 + flexible_bonus - overflex_penalty,
            -abs(shape.area - 16),
            -shape.fragile_count,
            -shape.id,
        )

    kept: list[ideal.ShapeRecord] = []
    kept_ids: set[int] = set()
    placement_budget = 0
    for shape in sorted(shapes, key=shape_key, reverse=True):
        count = placements_by_shape[shape.id]
        if count == 0:
            continue
        if len(kept) >= args.max_solver_shapes:
            break
        if placement_budget + count > args.max_solver_placements and len(kept) >= args.pieces:
            continue
        kept.append(shape)
        kept_ids.add(shape.id)
        placement_budget += count

    kept_placements = [placement for placement in placements if placement.shape_id in kept_ids]
    return kept, kept_placements


def profile_args(base_args: argparse.Namespace, profile: dict[str, object]) -> argparse.Namespace:
    args = copy.copy(base_args)
    for key, value in profile.items():
        setattr(args, key, value)
    return args


def board_limited_args(
    args: argparse.Namespace,
    base_args: argparse.Namespace,
    started_at: float,
) -> argparse.Namespace | None:
    elapsed = time.monotonic() - started_at
    remaining = base_args.total_time_limit - elapsed
    if remaining <= 5:
        return None

    board_args = copy.copy(args)
    requested = args.library_time_limit + args.solve_time_limit
    board_budget = min(requested, remaining)
    if board_budget < 2:
        return None

    library_budget = min(args.library_time_limit, max(1.0, board_budget * 0.45))
    solve_budget = min(args.solve_time_limit, max(1.0, board_budget - library_budget))
    board_args.library_time_limit = library_budget
    board_args.solve_time_limit = solve_budget
    return board_args


def search_profile(
    base_args: argparse.Namespace,
    profile: dict[str, object],
    started_at: float,
    results: list[RankedResult],
    near_misses: list[NearMiss],
) -> None:
    args = profile_args(base_args, profile)
    profile_name = str(profile["name"])
    macro_boards = ideal.generate_near_rect_macro_boards(args)
    if args.verbose:
        print(f"[{profile_name}] boards={len(macro_boards)}", file=sys.stderr, flush=True)

    for board_index, macro_board in enumerate(macro_boards, start=1):
        if time.monotonic() - started_at > base_args.total_time_limit:
            return
        if len(results) >= base_args.keep_candidates and not base_args.keep_searching:
            return

        board_args = board_limited_args(args, base_args, started_at)
        if board_args is None:
            return
        board_args.profile_name = profile_name
        board_args.board_index = board_index

        board = ideal.macro_to_small_board(macro_board)
        if args.verbose:
            print(
                f"[{profile_name}] board {board_index}/{len(macro_boards)} "
                f"macro_area={len(macro_board)} "
                f"lib={board_args.library_time_limit:.0f}s solve={board_args.solve_time_limit:.0f}s",
                file=sys.stderr,
                flush=True,
            )

        shapes = ideal.build_shape_library(board_args, macro_board)
        if args.verbose:
            print(f"[{profile_name}] built shapes={len(shapes)}", file=sys.stderr, flush=True)
        if len(shapes) < board_args.pieces:
            continue

        placements = ideal.enumerate_shape_placements(board, shapes)
        if args.verbose:
            print(f"[{profile_name}] enumerated placements={len(placements)}", file=sys.stderr, flush=True)
        if not placements:
            continue

        before_shapes = len(shapes)
        before_placements = len(placements)
        shapes, placements = trim_solver_input(shapes, placements, board_args)
        if len(shapes) < board_args.pieces or not placements:
            continue

        if args.verbose:
            print(
                f"[{profile_name}] solver input shapes={len(shapes)}/{before_shapes} "
                f"placements={len(placements)}/{before_placements}",
                file=sys.stderr,
                flush=True,
            )

        selected_sets = ideal.solve_with_cpsat_candidates(
            board,
            shapes,
            placements,
            board_args,
            max_candidates=board_args.solver_candidates_per_board,
        )
        if not selected_sets and board_args.single_cover_fallback:
            one_cover_args = copy.copy(board_args)
            one_cover_args.required_solutions = 1
            one_cover_args.solve_time_limit = min(
                board_args.solve_time_limit,
                board_args.single_cover_solve_time_limit,
            )
            selected_sets = ideal.solve_with_cpsat_candidates(
                board,
                shapes,
                placements,
                one_cover_args,
                max_candidates=board_args.solver_candidates_per_board,
            )
        if args.verbose:
            print(f"[{profile_name}] solver returned {len(selected_sets)} set(s)", file=sys.stderr, flush=True)
        for selected in selected_sets:
            candidate, near_miss, reject_reason = make_candidate(board, selected, board_args)
            if candidate is None:
                add_near_miss(near_misses, near_miss, base_args)
                if args.verbose:
                    print(
                        f"[{profile_name}] rejected: {reject_reason}",
                        file=sys.stderr,
                        flush=True,
                    )
                continue

            rank_score, notes = rank_candidate(candidate, profile_name)
            results.append(
                RankedResult(
                    candidate=candidate,
                    board_index=board_index,
                    profile_name=profile_name,
                    rank_score=rank_score,
                    notes=notes,
                )
            )
            results.sort(key=lambda result: result.rank_score, reverse=True)
            if args.verbose:
                print(
                    f"accepted score={rank_score:.1f} "
                    f"solutions={candidate.solution_count} notes={' / '.join(notes)}",
                    file=sys.stderr,
                    flush=True,
                )
            if len(results) >= base_args.keep_candidates and not base_args.keep_searching:
                return


def effort_caps(effort: str) -> list[int]:
    if effort == "relaxed":
        return [1, 8, 12, 16, 16, 10, 14]
    if effort == "fast":
        return [1, 3, 5, 5, 1, 4, 4]
    if effort == "deep":
        return [1, 20, 30, 30, 1, 30, 30]
    if effort == "extreme":
        return [1, 60, 90, 90, 1, 80, 80]
    return [1, 8, 12, 12, 1, 12, 12]


def build_profiles(args: argparse.Namespace) -> list[dict[str, object]]:
    caps = effort_caps(args.effort)
    if args.effort == "relaxed":
        specs = [
            ("6x4_full_half1_sol2", 6, 4, 0, 0, caps[0], 1, 2, args.shape_copies),
            ("5x5_full_half1_sol2", 5, 5, 0, 0, caps[1], 1, 2, args.shape_copies),
            ("5x5_remove0-2_half1_sol2", 5, 5, 0, 2, caps[2], 1, 2, args.shape_copies),
            ("5x5_remove0-3_half1_sol2", 5, 5, 0, 3, caps[3], 1, 2, args.shape_copies),
            ("6x5_remove5-6_half1_sol2", 6, 5, 5, 6, caps[4], 1, 2, args.shape_copies),
            ("6x4_full_half2_sol2", 6, 4, 0, 0, caps[5], 2, 2, args.shape_copies),
            ("5x5_remove0-3_half2_sol2", 5, 5, 0, 3, caps[6], 2, 2, args.shape_copies),
        ]
    else:
        specs = [
        ("5x5_full_half3_sol4", 5, 5, 0, 0, caps[0], 3, 4, 1),
        ("5x5_remove0-2_half3_sol4", 5, 5, 0, 2, caps[1], 3, 4, 1),
        ("5x5_remove0-3_half3_sol3", 5, 5, 0, 3, caps[2], 3, 3, 1),
        ("5x5_remove0-3_half2_sol2", 5, 5, 0, 3, caps[3], 2, 2, args.shape_copies),
        ("6x4_full_half2_sol4", 6, 4, 0, 0, caps[4], 2, 4, 1),
        ("5x5_remove0-3_half2_sol4", 5, 5, 0, 3, caps[5], 2, 4, 1),
        ("6x5_remove5-6_half2_sol4", 6, 5, 5, 6, caps[6], 2, 4, 1),
        ]
    profiles: list[dict[str, object]] = []
    for (
        name,
        board_w,
        board_h,
        remove_min,
        remove_max,
        max_boards,
        min_half,
        required_solutions,
        shape_copies,
    ) in specs:
        profiles.append(
            {
                "name": name,
                "board_w_macro": board_w,
                "board_h_macro": board_h,
                "board_remove_min": remove_min,
                "board_remove_max": remove_max,
                "max_board_candidates": max_boards,
                "min_half_cells": min_half,
                "required_solutions": required_solutions,
                "library_target": args.library_target,
                "max_fragile": 0,
                "allow_identical_pieces": args.allow_identical_pieces,
                "shape_copies": shape_copies if args.allow_identical_pieces else 1,
            }
        )
    return profiles


def write_ranked_outputs(results: list[RankedResult], output_dir: Path, limit: int) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    candidates = [result.candidate for result in results[:limit]]
    for result in results[:limit]:
        result.candidate.score = result.rank_score
    base.write_outputs(candidates, output_dir)

    summary = []
    for i, result in enumerate(results[:limit], start=1):
        metrics = small_board_metrics(result.candidate.board)
        summary.append(
            {
                "rank": i,
                "score": result.rank_score,
                "profile": result.profile_name,
                "board_index": result.board_index,
                "board_macro_area": metrics["macro_area"],
                "board_bbox": [metrics["bbox_w"], metrics["bbox_h"]],
                "board_fill": metrics["fill"],
                "board_extra_perimeter": metrics["perimeter_extra"],
                "solution_count_effective_fixed": result.candidate.solution_count,
                "rotated_solution_count_raw": result.candidate.analysis.rotated_solution_count,
                "half_cell_count_per_piece": result.candidate.analysis.half_cell_count_per_piece,
                "total_half_cell_count": result.candidate.analysis.total_half_cell_count,
                "quarter_artifact_count": result.candidate.analysis.quarter_artifact_count,
                "fragile_artifact_count": result.candidate.analysis.fragile_artifact_count,
                "duplicate_piece_count": result.candidate.analysis.duplicate_piece_count,
                "notes": result.notes,
            }
        )
    (output_dir / "ranking.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def near_miss_to_json(miss: NearMiss, rank: int) -> dict[str, object]:
    analysis = miss.candidate.analysis
    return {
        "rank": rank,
        "reject_reason": miss.reject_reason,
        "raw_solution_count": miss.raw_solution_count,
        "effective_solution_count": miss.effective_solution_count,
        "proof_layer_count": miss.proof_layer_count,
        "proof_valid": miss.proof_valid,
        "proof_message": miss.proof_message,
        "quarter_artifact_count": analysis.quarter_artifact_count,
        "fragile_artifact_count": analysis.fragile_artifact_count,
        "duplicate_piece_count": analysis.duplicate_piece_count,
        "board_metrics": miss.board_metrics,
        "profile_name": miss.profile_name,
        "board_index": miss.board_index,
        "seed": miss.seed,
        "selected_shape_ids": miss.selected_shape_ids,
        "selected_shape_cells": [
            [list(cell) for cell in cells]
            for cells in miss.selected_shape_cells
        ],
        "placements_by_piece_counts": miss.placements_by_piece_counts,
        "half_cell_count_per_piece": analysis.half_cell_count_per_piece,
        "total_half_cell_count": analysis.total_half_cell_count,
    }


def write_near_miss_html(near_misses: list[NearMiss], path: Path) -> None:
    cards = []
    for index, miss in enumerate(near_misses, start=1):
        candidate = miss.candidate
        metrics = miss.board_metrics
        solution_svg = (
            base.svg_solution(candidate.board, candidate.solutions[0])
            if candidate.solutions
            else "<div class='empty'>No phase-preserving fixed solution counted.</div>"
        )
        cards.append(
            f"""
            <article class="card">
              <div class="badge">NEAR MISS / NOT FINAL</div>
              <h2>Near Miss {index}</h2>
              <p class="reason">{escape(miss.reject_reason)}</p>
              <div class="metrics">
                <span>raw {miss.raw_solution_count}</span>
                <span>effective {miss.effective_solution_count}</span>
                <span>proof layers {miss.proof_layer_count}</span>
                <span>proof {str(miss.proof_valid).lower()}</span>
                <span>fragile {candidate.analysis.fragile_artifact_count}</span>
                <span>quarter {candidate.analysis.quarter_artifact_count}</span>
                <span>duplicate {candidate.analysis.duplicate_piece_count}</span>
                <span>fill {metrics['fill']:.2f}</span>
                <span>perimeter+ {metrics['perimeter_extra']:.0f}</span>
              </div>
              <div class="grid">
                <section><h3>Board</h3>{base.svg_board(candidate.board)}</section>
                <section><h3>First Counted Solution</h3>{solution_svg}</section>
                <section class="pieces"><h3>Pieces</h3>{base.svg_pieces(candidate.pieces)}</section>
              </div>
              <pre>{escape(miss.proof_message)}</pre>
            </article>
            """
        )

    html = f"""<!doctype html>
<html lang="ja">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Near Miss Gallery</title>
  <style>
    body {{ margin: 0; font-family: Arial, sans-serif; background: #f6f4ee; color: #222; }}
    header {{ padding: 24px 28px; background: #24211d; color: #fff; }}
    header h1 {{ margin: 0 0 8px; font-size: 24px; }}
    header p {{ margin: 0; color: #ddd; }}
    main {{ padding: 20px; display: grid; gap: 18px; }}
    .card {{ border: 1px solid #d2cab8; background: #fff; border-radius: 8px; padding: 18px; }}
    .badge {{ display: inline-block; background: #8f1d1d; color: #fff; font-weight: 700; padding: 6px 9px; border-radius: 4px; }}
    h2 {{ margin: 12px 0 6px; font-size: 20px; }}
    h3 {{ margin: 0 0 8px; font-size: 15px; }}
    .reason {{ font-family: Consolas, monospace; background: #f4eee5; padding: 8px; border-radius: 4px; }}
    .metrics {{ display: flex; flex-wrap: wrap; gap: 8px; margin: 10px 0 16px; }}
    .metrics span {{ border: 1px solid #d7d0c2; border-radius: 4px; padding: 5px 8px; background: #fbfaf7; }}
    .grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); gap: 16px; align-items: start; }}
    svg {{ max-width: 100%; height: auto; border: 1px solid #ddd4c4; }}
    .pieces {{ grid-column: 1 / -1; }}
    .empty {{ min-height: 120px; display: grid; place-items: center; border: 1px dashed #cbbf9f; color: #6f624d; }}
    pre {{ white-space: pre-wrap; background: #28241f; color: #f2eadc; padding: 10px; border-radius: 4px; }}
  </style>
</head>
<body>
  <header>
    <h1>Near Miss Gallery</h1>
    <p>NEAR MISS / NOT FINAL. These candidates failed at least one acceptance rule.</p>
  </header>
  <main>
    {''.join(cards) if cards else '<p>No near misses were captured.</p>'}
  </main>
</body>
</html>
"""
    path.write_text(html, encoding="utf-8")


def write_near_miss_outputs(near_misses: list[NearMiss], args: argparse.Namespace) -> None:
    if not args.write_near_misses:
        return
    output_dir = args.near_miss_output_dir
    output_dir.mkdir(parents=True, exist_ok=True)
    limited = near_misses[: args.near_miss_limit]
    payload = [near_miss_to_json(miss, index) for index, miss in enumerate(limited, start=1)]
    (output_dir / "near_miss.json").write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    write_near_miss_html(limited, output_dir / "index.html")


def add_random_fallback_results(
    args: argparse.Namespace,
    started_at: float,
    results: list[RankedResult],
) -> None:
    if args.no_random_fallback:
        return
    if len(results) >= args.keep_candidates and not args.keep_searching:
        return

    remaining = args.total_time_limit - (time.monotonic() - started_at)
    fallback_time = min(args.random_fallback_time_limit, remaining)
    if fallback_time <= 10:
        return

    wanted = max(1, args.keep_candidates - len(results))
    fallback_args = argparse.Namespace(
        pieces=args.pieces,
        board_w=6,
        board_h=5,
        min_solutions=args.min_acceptable_solutions,
        max_solutions=args.solution_count_limit,
        allow_mirror=False,
        no_rotate=args.no_rotate,
        seed=args.seed + 10_000,
        candidates=wanted,
        time_limit=fallback_time,
        output_dir=args.output_dir,
        allow_holes=args.allow_holes,
        allow_identical_pieces=args.allow_identical_pieces,
        min_half_cells=args.fallback_min_half_cells,
        solution_count_limit=args.solution_count_limit,
        verbose=args.verbose,
    )
    if args.verbose:
        print(
            f"[random_fallback] time={fallback_time:.0f}s candidates={wanted}",
            file=sys.stderr,
            flush=True,
        )

    for candidate in base.generate_candidates(fallback_args):
        effective_solutions = unique_effective_solutions(
            candidate.solutions,
            candidate.pieces,
            limit=args.solution_count_limit,
        )
        if len(effective_solutions) < args.min_acceptable_solutions:
            continue
        rotated_count, _, _ = base.count_solutions(
            candidate.board,
            candidate.pieces,
            allow_rotate=not args.no_rotate,
            allow_mirror=False,
            limit=args.solution_count_limit,
        )
        analysis = base.analyze_candidate(
            candidate.board,
            candidate.pieces,
            effective_solutions,
            len(effective_solutions),
            rotated_count,
            candidate.placements_by_piece,
            args.min_acceptable_solutions,
            args.target_solutions,
        )
        if analysis.quarter_artifact_count != 0:
            continue
        if analysis.fragile_artifact_count != 0:
            continue
        if analysis.duplicate_piece_count != 0 and not args.allow_identical_pieces:
            continue

        candidate.solutions = effective_solutions
        candidate.solution_count = len(effective_solutions)
        candidate.analysis = analysis
        rank_score, notes = rank_candidate(candidate, "random_fallback")
        results.append(
            RankedResult(
                candidate=candidate,
                board_index=0,
                profile_name="random_fallback",
                rank_score=rank_score,
                notes=notes,
            )
        )
        results.sort(key=lambda result: result.rank_score, reverse=True)


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="One-shot ranked ideal search.")
    parser.add_argument("--pieces", type=int, default=6)
    parser.add_argument("--min-piece-area", type=int, default=14)
    parser.add_argument("--max-piece-area", type=int, default=18)
    parser.add_argument("--target-solutions", type=int, default=4)
    parser.add_argument("--min-acceptable-solutions", type=int, default=2)
    parser.add_argument("--solution-count-limit", type=int, default=200)
    parser.add_argument("--library-target", type=int, default=2500)
    parser.add_argument("--library-time-limit", type=float, default=180.0)
    parser.add_argument("--solve-time-limit", type=float, default=180.0)
    parser.add_argument("--total-time-limit", type=float, default=3600.0)
    parser.add_argument("--transfer-attempts", type=int, default=120)
    parser.add_argument("--max-per-rotational-family", type=int, default=3)
    parser.add_argument("--workers", type=int, default=8)
    parser.add_argument("--seed", type=int, default=1)
    parser.add_argument("--effort", choices=("relaxed", "fast", "balanced", "deep", "extreme"), default="relaxed")
    parser.add_argument("--output-dir", type=Path, default=Path("out_ranked"))
    parser.add_argument("--keep-candidates", type=int, default=12)
    parser.add_argument("--keep-searching", action="store_true")
    parser.add_argument("--allow-identical-pieces", action="store_true", default=True)
    parser.add_argument("--no-identical-pieces", dest="allow_identical_pieces", action="store_false")
    parser.add_argument("--shape-copies", type=int, default=2)
    parser.add_argument("--solver-candidates-per-board", type=int, default=3)
    parser.add_argument("--single-cover-fallback", action="store_true", default=True)
    parser.add_argument("--no-single-cover-fallback", dest="single_cover_fallback", action="store_false")
    parser.add_argument("--single-cover-solve-time-limit", type=float, default=45.0)
    parser.add_argument("--max-solver-shapes", type=int, default=2500)
    parser.add_argument("--max-solver-placements", type=int, default=22_000)
    parser.add_argument("--max-board-candidates-per-remove", type=int, default=2000)
    parser.add_argument("--random-fallback-time-limit", type=float, default=600.0)
    parser.add_argument("--fallback-min-half-cells", type=int, default=1)
    parser.add_argument("--no-random-fallback", action="store_true")
    parser.add_argument("--write-near-misses", action="store_true")
    parser.add_argument("--near-miss-limit", type=int, default=20)
    parser.add_argument("--accept-one-solution-nearmiss", action="store_true")
    parser.add_argument("--near-miss-output-dir", type=Path, default=Path("out_debug/near_miss"))
    parser.add_argument("--allow-holes", action="store_true")
    parser.add_argument("--no-rotate", action="store_true")
    parser.add_argument("--verbose", action="store_true")
    parser.add_argument("--solver-log", action="store_true")
    return parser.parse_args(argv)


def main(argv: list[str] | None = None) -> int:
    args = parse_args(argv)
    if ideal.cp_model is None:
        print("OR-Tools is not installed. Run: python -m pip install -r requirements.txt", file=sys.stderr)
        return 2

    started_at = time.monotonic()
    results: list[RankedResult] = []
    near_misses: list[NearMiss] = []
    for profile in build_profiles(args):
        if time.monotonic() - started_at > args.total_time_limit:
            break
        search_profile(args, profile, started_at, results, near_misses)
        if len(results) >= args.keep_candidates and not args.keep_searching:
            break

    add_random_fallback_results(args, started_at, results)
    write_ranked_outputs(results, args.output_dir, args.keep_candidates)
    write_near_miss_outputs(near_misses, args)
    if not results:
        if near_misses and args.write_near_misses:
            print(f"No ranked candidates found. Saved near misses to {args.near_miss_output_dir}", file=sys.stderr)
        else:
            print("No ranked candidates found. The output gallery is empty.", file=sys.stderr)
        return 1
    print(f"Saved {min(len(results), args.keep_candidates)} ranked candidate(s) to {args.output_dir}")
    print(f"Best score: {results[0].rank_score:.1f}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile verified_direct_search.py
#!/usr/bin/env python3
"""Direct verified search for robust half-cell puzzle candidates.

This script intentionally avoids CP-SAT.  Every generated six-piece set is
checked by the Python exact-cover counter before it is accepted.  It is slower
than ranked_ideal_search.py, but easier to trust and easier to inspect.
"""

from __future__ import annotations

import argparse
import copy
import random
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import generate_half_polyomino as base
import ideal_half_polyomino_search as ideal
import ranked_ideal_search as ranked


@dataclass
class SearchStats:
    attempts: int = 0
    generated_piece_sets: int = 0
    raw_solution_hits: int = 0
    effective_solution_hits: int = 0
    rejected_fragile: int = 0
    rejected_quarter: int = 0
    rejected_too_few_solutions: int = 0
    accepted: int = 0


def near_rect_boards(
    width: int,
    height: int,
    remove_min: int,
    remove_max: int,
    args: argparse.Namespace,
) -> list[set[tuple[int, int]]]:
    board_args = argparse.Namespace(
        board_w_macro=width,
        board_h_macro=height,
        board_remove_min=remove_min,
        board_remove_max=remove_max,
        max_board_candidates=args.max_boards_per_profile,
        max_board_candidates_per_remove=5000,
        allow_holes=args.allow_holes,
        pieces=args.pieces,
        min_piece_area=args.min_piece_area,
        max_piece_area=args.max_piece_area,
    )
    return ideal.generate_near_rect_macro_boards(board_args)


def build_board_profiles(args: argparse.Namespace) -> list[tuple[str, list[set[tuple[int, int]]]]]:
    profiles = [
        ("6x4_full", [ideal.macro_rectangle(6, 4)]),
        ("5x5_remove1", near_rect_boards(5, 5, 1, 1, args)),
        ("5x5_remove2", near_rect_boards(5, 5, 2, 2, args)),
        ("6x5_remove5-6", near_rect_boards(6, 5, 5, 6, args)),
    ]
    return [(name, boards) for name, boards in profiles if boards]


def compact_random_board(rng: random.Random, args: argparse.Namespace) -> set[tuple[int, int]] | None:
    area = rng.choice((24, 25))
    return base.random_macro_board(
        rng,
        width=args.board_w,
        height=args.board_h,
        area=area,
        allow_holes=args.allow_holes,
        max_tries=200,
    )


def make_covering_piece_set(
    regions: list[set[tuple[int, int]]],
    rng: random.Random,
    args: argparse.Namespace,
) -> list[set[tuple[int, int]]] | None:
    edges = ideal.boundary_edges(regions)
    if not edges:
        return None

    for _ in range(args.transfer_attempts):
        masks_by_piece = [{cell: base.MASK_FULL for cell in region} for region in regions]
        areas = [len(region) * 4 for region in regions]
        split_cells: set[tuple[int, int]] = set()
        shuffled_edges = edges[:]
        rng.shuffle(shuffled_edges)
        split_budget = rng.randint(
            max(1, args.min_half_cells * len(regions) // 2),
            max(1, len(shuffled_edges)),
        )

        for c, d, p, q, direction in shuffled_edges:
            if len(split_cells) >= split_budget:
                break
            options = [(c, p, q, True), (d, q, p, False)]
            rng.shuffle(options)
            for cell, donor, receiver, positive in options:
                if cell in split_cells:
                    continue
                if areas[donor] - 2 < args.min_piece_area:
                    continue
                if areas[receiver] + 2 > args.max_piece_area:
                    continue
                donor_mask, receiver_mask = ideal.transfer_masks(direction, positive)
                masks_by_piece[donor][cell] = donor_mask
                masks_by_piece[receiver][cell] = receiver_mask
                areas[donor] -= 2
                areas[receiver] += 2
                split_cells.add(cell)
                break

        pieces = [ideal.align_cells(base.masks_to_cells(masks)) for masks in masks_by_piece]
        if all(
            ideal.validate_ideal_piece(
                piece,
                min_area=args.min_piece_area,
                max_area=args.max_piece_area,
                min_half_cells=args.min_half_cells,
                max_fragile=0,
            )
            for piece in pieces
        ):
            return pieces
    return None


def generate_piece_set_for_board(
    rng: random.Random,
    macro_board: set[tuple[int, int]],
    args: argparse.Namespace,
) -> list[set[tuple[int, int]]] | None:
    regions = ideal.random_tetromino_partition(rng, macro_board, args.pieces)
    if regions is None:
        return None
    return make_covering_piece_set(regions, rng, args)


def evaluate_piece_set(
    macro_board: set[tuple[int, int]],
    pieces: list[set[tuple[int, int]]],
    profile_name: str,
    attempt: int,
    args: argparse.Namespace,
) -> tuple[base.Candidate, int, int]:
    board = ideal.macro_to_small_board(macro_board)
    min_x, min_y, _, _ = base.bounds(board)
    board = {(x - min_x, y - min_y) for x, y in board}
    pieces = [ideal.align_cells(piece) for piece in pieces]
    raw_count, raw_solutions, placements_by_piece = ideal.count_solutions_fixed_phase(
        board,
        pieces,
        limit=args.solution_count_limit,
    )
    effective_solutions = ranked.unique_effective_solutions(
        raw_solutions,
        pieces,
        limit=args.solution_count_limit,
    )
    effective_count = len(effective_solutions)
    rotated_count, _, _ = base.count_solutions(
        board,
        pieces,
        allow_rotate=not args.no_rotate,
        allow_mirror=False,
        limit=args.solution_count_limit,
    )
    analysis = base.analyze_candidate(
        board,
        pieces,
        effective_solutions,
        effective_count,
        rotated_count,
        placements_by_piece,
        args.min_solutions,
        args.max_solutions,
    )
    candidate = base.Candidate(
        board=board,
        pieces=pieces,
        solutions=effective_solutions,
        solution_count=effective_count,
        placements_by_piece=placements_by_piece,
        score=analysis.difficulty_score,
        analysis=analysis,
        attempts=attempt,
    )
    score, _notes = ranked.rank_candidate(candidate, profile_name)
    candidate.score = score
    return candidate, raw_count, effective_count


def candidate_key(candidate: base.Candidate) -> str:
    return repr(
        (
            sorted(candidate.board),
            [sorted(piece) for piece in candidate.pieces],
        )
    )


def save_candidates_incremental(candidates: list[base.Candidate], output_dir: Path) -> None:
    candidates.sort(key=lambda candidate: candidate.score, reverse=True)
    base.write_outputs(candidates, output_dir)


def make_near_miss(
    candidate: base.Candidate,
    raw_count: int,
    effective_count: int,
    reject_reason: str,
    profile_name: str,
    args: argparse.Namespace,
) -> ranked.NearMiss:
    return ranked.NearMiss(
        candidate=candidate,
        reject_reason=reject_reason,
        raw_solution_count=raw_count,
        effective_solution_count=effective_count,
        proof_layer_count=0,
        proof_valid=False,
        proof_message="direct search candidate; no CP-SAT proof",
        profile_name=profile_name,
        board_index=0,
        seed=args.seed,
        selected_shape_ids=list(range(len(candidate.pieces))),
        selected_shape_cells=[sorted(piece) for piece in candidate.pieces],
        board_metrics=ranked.small_board_metrics(candidate.board),
        placements_by_piece_counts=[len(placements) for placements in candidate.placements_by_piece],
    )


def log_stats(stats: SearchStats) -> None:
    print(
        " ".join(
            [
                f"attempts={stats.attempts}",
                f"generated_piece_sets={stats.generated_piece_sets}",
                f"raw_solution_hits={stats.raw_solution_hits}",
                f"effective_solution_hits={stats.effective_solution_hits}",
                f"rejected_fragile={stats.rejected_fragile}",
                f"rejected_quarter={stats.rejected_quarter}",
                f"rejected_too_few_solutions={stats.rejected_too_few_solutions}",
                f"accepted={stats.accepted}",
            ]
        ),
        file=sys.stderr,
        flush=True,
    )


def search(args: argparse.Namespace) -> tuple[list[base.Candidate], list[ranked.NearMiss]]:
    rng = random.Random(args.seed)
    start = time.monotonic()
    profiles = build_board_profiles(args)
    candidates: list[base.Candidate] = []
    near_misses: list[ranked.NearMiss] = []
    seen: set[str] = set()
    stats = SearchStats()

    args.output_dir.mkdir(parents=True, exist_ok=True)
    while len(candidates) < args.candidates and time.monotonic() - start < args.time_limit:
        stats.attempts += 1
        profile_name: str
        macro_board: set[tuple[int, int]] | None
        if stats.attempts % 5 == 0:
            profile_name = "compact_random"
            macro_board = compact_random_board(rng, args)
        else:
            profile_name, boards = profiles[(stats.attempts - 1) % len(profiles)]
            macro_board = rng.choice(boards)

        if macro_board is None:
            continue
        pieces = generate_piece_set_for_board(rng, macro_board, args)
        if pieces is None:
            if stats.attempts % 25 == 0:
                log_stats(stats)
            continue

        stats.generated_piece_sets += 1
        candidate, raw_count, effective_count = evaluate_piece_set(
            macro_board,
            pieces,
            profile_name,
            stats.attempts,
            args,
        )

        if candidate.analysis.quarter_artifact_count != 0:
            stats.rejected_quarter += 1
            continue
        if candidate.analysis.fragile_artifact_count != 0:
            stats.rejected_fragile += 1
            continue
        if raw_count > 0:
            stats.raw_solution_hits += 1
        if effective_count > 0:
            stats.effective_solution_hits += 1

        reject_reason = ""
        if effective_count < args.min_solutions:
            stats.rejected_too_few_solutions += 1
            reject_reason = f"effective_solution_count={effective_count}<min={args.min_solutions}"
        elif not args.allow_identical_pieces and candidate.analysis.duplicate_piece_count != 0:
            reject_reason = f"duplicate_piece_count={candidate.analysis.duplicate_piece_count}"
        elif effective_count > args.max_solutions:
            reject_reason = f"effective_solution_count={effective_count}>max={args.max_solutions}"

        key = candidate_key(candidate)
        if reject_reason:
            miss = make_near_miss(candidate, raw_count, effective_count, reject_reason, profile_name, args)
            ranked.add_near_miss(near_misses, miss, args)
        elif key not in seen:
            seen.add(key)
            candidates.append(candidate)
            stats.accepted += 1
            save_candidates_incremental(candidates, args.output_dir)
            if args.verbose:
                print(
                    f"accepted #{stats.accepted} attempt={stats.attempts} "
                    f"profile={profile_name} effective={effective_count} score={candidate.score:.1f}",
                    file=sys.stderr,
                    flush=True,
                )

        if args.write_near_misses and near_misses and stats.attempts % args.near_miss_save_interval == 0:
            ranked.write_near_miss_outputs(near_misses, args)
        if stats.attempts % 25 == 0:
            log_stats(stats)

    if candidates:
        save_candidates_incremental(candidates, args.output_dir)
    ranked.write_near_miss_outputs(near_misses, args)
    return candidates, near_misses


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Direct verified half-cell search.")
    parser.add_argument("--pieces", type=int, default=6)
    parser.add_argument("--min-half-cells", type=int, default=1)
    parser.add_argument("--min-solutions", type=int, default=2)
    parser.add_argument("--max-solutions", type=int, default=100)
    parser.add_argument("--allow-identical-pieces", action="store_true", default=True)
    parser.add_argument("--no-identical-pieces", dest="allow_identical_pieces", action="store_false")
    parser.add_argument("--board-w", type=int, default=6)
    parser.add_argument("--board-h", type=int, default=5)
    parser.add_argument("--time-limit", type=float, default=1800.0)
    parser.add_argument("--candidates", type=int, default=20)
    parser.add_argument("--seed", type=int, default=1001)
    parser.add_argument("--output-dir", type=Path, default=Path("out_direct"))
    parser.add_argument("--min-piece-area", type=int, default=14)
    parser.add_argument("--max-piece-area", type=int, default=18)
    parser.add_argument("--solution-count-limit", type=int, default=100)
    parser.add_argument("--transfer-attempts", type=int, default=120)
    parser.add_argument("--max-boards-per-profile", type=int, default=24)
    parser.add_argument("--allow-holes", action="store_true")
    parser.add_argument("--no-rotate", action="store_true")
    parser.add_argument("--verbose", action="store_true")
    parser.add_argument("--write-near-misses", action="store_true", default=True)
    parser.add_argument("--near-miss-limit", type=int, default=20)
    parser.add_argument("--accept-one-solution-nearmiss", action="store_true", default=True)
    parser.add_argument("--near-miss-output-dir", type=Path, default=Path("out_direct/near_miss"))
    parser.add_argument("--near-miss-save-interval", type=int, default=25)
    return parser.parse_args(argv)


def validate_args(args: argparse.Namespace) -> None:
    if args.pieces != 6:
        raise SystemExit("verified direct search currently expects --pieces 6")
    if args.solution_count_limit < args.min_solutions:
        raise SystemExit("--solution-count-limit must be at least --min-solutions")
    if args.min_piece_area % 2 or args.max_piece_area % 2:
        raise SystemExit("piece area bounds must be even small-cell counts")


def main(argv: list[str] | None = None) -> int:
    args = parse_args(argv)
    validate_args(args)
    candidates, near_misses = search(args)
    if candidates:
        print(f"Saved {len(candidates)} accepted candidate(s) to {args.output_dir}")
        return 0
    if near_misses:
        print(f"No accepted candidates. Saved near misses to {args.near_miss_output_dir}", file=sys.stderr)
        return 1
    print("No accepted candidates or near misses found.", file=sys.stderr)
    return 1


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
!pip install -r requirements.txt

In [ ]:
# Quick verification
import ortools
import ideal_half_polyomino_search as ideal
print("ortools", ortools.__version__)
print("near-rect board generator", hasattr(ideal, "generate_near_rect_macro_boards"))
print("fixed phase counter", hasattr(ideal, "count_solutions_fixed_phase"))

In [ ]:
# Debug-first ranked run with near misses
!python ranked_ideal_search.py \
  --effort relaxed \
  --total-time-limit 900 \
  --library-target 800 \
  --library-time-limit 40 \
  --solve-time-limit 45 \
  --solver-candidates-per-board 12 \
  --max-solver-shapes 3000 \
  --max-solver-placements 50000 \
  --random-fallback-time-limit 120 \
  --fallback-min-half-cells 1 \
  --workers 2 \
  --keep-candidates 12 \
  --seed 303 \
  --output-dir out_debug \
  --write-near-misses \
  --near-miss-limit 20 \
  --accept-one-solution-nearmiss \
  --verbose

In [ ]:
# Direct verified search, no CP-SAT
!python verified_direct_search.py \
  --min-half-cells 1 \
  --min-solutions 2 \
  --max-solutions 100 \
  --allow-identical-pieces \
  --board-w 6 \
  --board-h 5 \
  --time-limit 1800 \
  --candidates 20 \
  --seed 1001 \
  --output-dir out_direct \
  --verbose

In [ ]:
# Zip and download ranked debug results
!zip -r out_debug.zip out_debug
from google.colab import files
files.download("out_debug.zip")

In [ ]:
# Zip and download direct results
!zip -r out_direct.zip out_direct
from google.colab import files
files.download("out_direct.zip")

In [ ]:
# Longer deep ranked run, if you want to keep searching
# !python ranked_ideal_search.py \
#   --effort deep \
#   --library-target 5000 \
#   --library-time-limit 600 \
#   --solve-time-limit 600 \
#   --single-cover-solve-time-limit 120 \
#   --total-time-limit 14400 \
#   --solver-candidates-per-board 6 \
#   --max-solver-shapes 4000 \
#   --max-solver-placements 40000 \
#   --keep-candidates 20 \
#   --keep-searching \
#   --workers 2 \
#   --output-dir out_ranked_deep \
#   --write-near-misses \
#   --near-miss-output-dir out_ranked_deep/near_miss \
#   --verbose